# **Problem Description**


**Overview**

This dataset contains records for 1,000 students, capturing a holistic view of academic routines, lifestyle choices, socio-demographic backgrounds, and performance metrics. Designed primarily for **supervised classification** and **predictive modeling**, it enables granular feature importance evaluation and exploratory data analysis (EDA) to understand how controllable habits and uncontrollable external factors interact.

Key Data Categories & Attributes

* **Academic Routines & Engagement (Controllable)**
* **Study Hours:** Weekly dedicated study time (continuous).
* **Class Attendance:** Percentage of classes attended (continuous/percentage).
* **Assignment Completion:** On-time submission rate (percentage or ordinal).


* **Lifestyle & Wellbeing Factors**
* **Sleep Duration:** Average daily sleep hours (continuous).
* **Extracurricular Involvement:** Weekly hours in sports, clubs, or arts (continuous/categorical).
* **Part-Time Employment:** Weekly work hours (continuous/binary indicator).
* **Stress Level:** Self-reported stress index (ordinal scale: Low, Medium, High).


* **Demographics & Background (Non-Controllable)**
* **Gender:** Self-identified gender category.
* **Parental Education Level:** Highest degree attained by parents (ordinal).
* **Household Income Band:** Socio-economic status proxy (categorical/ordinal).
* **Internet Access at Home:** Reliability indicator (binary).


* **Target Variables (Academic Outcomes)**
* **Grade Classification (Primary Target):** Final letter grade (`A`, `B`, `C`, `D`, `F` or binary `Pass`/`Fail`).
* **Final Exam Score (Secondary Target):** Continuous numerical score (0–100) for regression baselines or binning experiments.



---

Primary Use Cases & Modeling Frameworks

* **Multi-Class Classification:** Predicting letter grades (`A`–`F`) or pass/fail risk status using tree-based ensembles (Random Forest, XGBoost) and logistic models.
* **Feature Importance & Driver Analysis:** Identifying whether external socio-demographic factors outweigh controllable study habits using SHAP values and permutation importance.
* **Exploratory Data Analysis (EDA):** Mapping non-linear interactions (e.g., how part-time work impacts sleep and exam performance across different income brackets).
* **Regression Benchmarking:** Predicting exact exam scores to evaluate continuous metrics (RMSE, $R^2$) prior to grade binning.

---

# **Setup**

In [1]:
# Core Data Libraries
import os
import json
import joblib
import numpy as np
import pandas as pd

# Visualization
import plotly.express as px
from plotly.offline import init_notebook_mode

# Scikit-Learn Preprocessing & Pipeline Elements
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder, RobustScaler, PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.base import clone

# Model Evaluation & Cross Validation
from sklearn.model_selection import KFold, cross_validate
from sklearn.metrics import (
    root_mean_squared_error, 
    mean_absolute_error, 
    r2_score, 
    accuracy_score, 
    precision_recall_fscore_support
)

# Traditional ML Regressors
from sklearn.linear_model import (
    LinearRegression, 
    Ridge, 
    Lasso, 
    ElasticNet, 
    HuberRegressor, 
    BayesianRidge
)
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR

# Tree-Based Regressors & Ensembles
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestRegressor, 
    ExtraTreesRegressor, 
    VotingRegressor
)

# Advanced Gradient Boosting Libraries
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor

In [2]:
# CONSTANTS
DF_PATH = "/kaggle/input/datasets/harshadapatil31/student-performance-and-study-habits-dataset/student_performance_dataset.csv"
TARGET_VAR = 'final_grade'

In [3]:
# Utility functions
def printli(items:list, *, apply=None)->None:
    for index, item in enumerate(items, start=1):
        if apply is not None:
            print(f"{index}. {apply(item)}")
        else:
            print(f"{index}. {item}")

# **Data**

## **Data loading**

In [4]:
df = pd.read_csv(DF_PATH)
df.head()

,student_id,gender,study_time_hours,attendance_percent,sleep_hours,parental_education,internet_access,extracurricular_activities,part_time_job,previous_grade,final_exam_score,final_grade
0,1,Male,4.0,98.0,6.5,Bachelors,Yes,Yes,No,76.9,100.0,A
1,2,Female,6.3,100.0,5.7,High School,Yes,Yes,Yes,75.5,100.0,A
2,3,Male,4.9,85.3,7.9,Bachelors,Yes,No,Yes,88.5,97.3,A
3,4,Male,2.6,77.5,8.0,NaN,Yes,Yes,No,85.1,83.8,B
4,5,Male,2.2,89.6,4.6,Bachelors,Yes,No,Yes,61.8,68.3,D


## **Data Profiling**

In [5]:
df.shape

(1000, 12)

At 1,000 rows and 12 total variables (11 features and 1 classification target), this dataset is small compared to modern large-scale benchmarks. Because train-test splitting reduces the sample size further, high-capacity gradient boosting algorithms like XGBoost risk overfitting.

To maintain strong generalization, simpler models such as Logistic Regression, Support Vector Machines, Decision Trees, or Random Forests are better choices.

The 12 variables consist of 11 predictor features—covering study habits, lifestyle, and demographics—and 1 categorical target variable for grade classification. Let's examine each feature individually to understand its role.

In [6]:
column_names = list(df.columns)
feature_names = column_names[:-1]

printli(feature_names, apply=lambda x: " ".join(x.split("_")).title())

1. Student Id
2. Gender
3. Study Time Hours
4. Attendance Percent
5. Sleep Hours
6. Parental Education
7. Internet Access
8. Extracurricular Activities
9. Part Time Job
10. Previous Grade
11. Final Exam Score


While the features are fairly intuitive, examining their structure is crucial before modeling. Out of the 11 variables, `Student ID` is purely an identifier with no predictive value, so it should be dropped during preprocessing.

The remaining 10 features—`Gender`, `Study Time`, `Attendance Percentage`, `Sleep Hours`, `Parental Education`, `Internet Access`, `Extracurricular Activities`, `Part-Time Job`, `Previous Grade`, and `Final Exam Score`—warrant close inspection.

Beyond knowing what each feature represents, we need to understand *how* it is stored. Checking data types, numerical scales, and categorical unique values ensures proper encoding (e.g., one-hot vs. ordinal) and prevents data leakage.

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 12 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   student_id                  1000 non-null   int64  
 1   gender                      1000 non-null   object 
 2   study_time_hours            1000 non-null   float64
 3   attendance_percent          1000 non-null   float64
 4   sleep_hours                 1000 non-null   float64
 5   parental_education          898 non-null    object 
 6   internet_access             1000 non-null   object 
 7   extracurricular_activities  1000 non-null   object 
 8   part_time_job               1000 non-null   object 
 9   previous_grade              1000 non-null   float64
 10  final_exam_score            1000 non-null   float64
 11  final_grade                 1000 non-null   object 
dtypes: float64(5), int64(1), object(6)
memory usage: 93.9+ KB


A review of the dataset info highlights a key challenge: `Parental Education` contains 102 missing values (roughly 10% of our 1,000-row sample). While 900 complete rows remain, dropping missing entries isn't ideal for a dataset this small. Because this feature is categorical/ordinal, simple numerical techniques don't apply directly—imputation will require mode filling, creating an "Unknown" category, or using predictive models.

Modern algorithms like Random Forests and XGBoost handle missing categorical data natively, allowing for a streamlined baseline pipeline using raw data. Traditional models (e.g., SVMs, Logistic Regression), however, require preprocessed data with explicit imputation and encoding (such as ordinal or target encoding).

Excluding the target, the dataset features a balanced 50/50 split: 6 numerical features and 5 categorical features, along with 1 categorical target variable (`Final Grade`), confirming this as a classification task. Aside from `Parental Education`, all other features are complete with zero null values. Let's examine each column individually.

In [8]:
cat_features = list(df.select_dtypes(include='object').columns)
num_features = list(df.select_dtypes(exclude='object').columns)

In [9]:
print('Categorical features: ')
for index, name in enumerate(cat_features, start=1):
    print(f'{index}. {" ".join(name.split("_")).title():30} {df[name].nunique():3} {df[name].unique()}')

Categorical features: 
1. Gender                           2 ['Male' 'Female']
2. Parental Education               4 ['Bachelors' 'High School' nan 'Masters' 'PhD']
3. Internet Access                  2 ['Yes' 'No']
4. Extracurricular Activities       2 ['Yes' 'No']
5. Part Time Job                    2 ['No' 'Yes']
6. Final Grade                      5 ['A' 'B' 'D' 'C' 'F']


Looking at the categorical features, several intuitive patterns and modeling implications emerge:

* **Parental Education:** Contains inherent hierarchy (*High School < Bachelor's < Master's < PhD*). While intuitive to assume higher parental education correlates with better student performance, EDA will test if this bias holds true in the data. Since the categories carry a natural order, we can use simple **Ordinal Encoding** (1 to 4) rather than complex target or M-estimate encodings.
* **Internet Access & Extracurricular Activities:** Both are binary (*Yes/No*). While extracurriculars involve a trade-off between study time and personal growth, internet access presents a unique limitation—access alone doesn't reflect actual study usage vs. entertainment. Measure of daily usage would have been more informative.
* **Part-Time Job:** Another binary factor affecting time management. While working reduces available study hours, it often fosters discipline, so its net impact on performance will be compelling to analyze.
* **Gender & Final Grade:** `Gender` serves as a standard demographic variable, while `Final Grade` (*A* through *F*) acts as our ordinal target variable for classification.

In [10]:
print('Numerical features: ')
for index, name in enumerate(num_features, start=1):
    print(f'{index}. {" ".join(name.split("_")).title():30} Sample Var: {df[name].var():5.2f}')

Numerical features: 
1. Student Id                     Sample Var: 83416.67
2. Study Time Hours               Sample Var:  2.19
3. Attendance Percent             Sample Var: 85.95
4. Sleep Hours                    Sample Var:  1.45
5. Previous Grade                 Sample Var: 159.10
6. Final Exam Score               Sample Var: 106.94


Analyzing the numerical features using variance gives key insights into data distribution and feature utility, though variance must be interpreted carefully:

* **Variance Interpretation:** Zero or low variance means the feature is virtually constant, offering little predictive power. Extremely high variance often points to unique identifiers or high dispersion. Features in the "sweet spot" of moderate variance contain rich signal for modeling.
* **Low-Variance Features (`Study Hours`, `Sleep Hours`):** The low variance here indicates that most students follow nearly identical routines. Because these patterns lack distinct variation, they may contribute less to differentiating academic outcomes.
* **Target Leakage (`Final Exam Score`):** The final score directly determines the categorical target grade. Keeping `Final Exam Score` as an input feature during classification would cause severe data leakage.
* **Dual-Modeling Strategy:** Rather than direct classification, an effective pipeline involves treating `Final Exam Score` as a continuous target for **regression**, then binning those predicted scores into letter grades for classification.
* **Moderate-Variance Features (`Attendance Percentage`, `Previous Grade`):** These show healthy spread across the student population, making them promising primary predictors for exam performance.

In [11]:
relation_df = df[[num_features[-1], TARGET_VAR]]
relation_df.groupby(TARGET_VAR).describe()

final_exam_score                                           \
                       count       mean       std   min     25%   50%   
final_grade                                                             
A                      284.0  96.069366  3.401851  90.0  93.075  96.3   
B                      354.0  84.752825  2.930847  80.0  82.200  84.8   
C                      261.0  75.413410  2.809793  70.0  73.100  75.5   
D                       89.0  66.228090  2.716239  60.0  63.700  66.7   
F                       12.0  56.675000  3.711193  46.8  56.500  57.7   

                             
                 75%    max  
final_grade                  
A            100.000  100.0  
B             87.300   89.9  
C             77.900   79.9  
D             68.600   69.9  
F             58.925   59.7

This pivot aligns with the underlying data structure. The final exam scores—whether raw points scaled to 100 or normalized percentages—follow a deterministic grading scale (e.g., 90–100 is an `A`, 80–89 is a `B`).

Attempting to classify letter grades directly adds unnecessary complexity and loses fine-grained numerical information. A two-stage pipeline is far more robust:

* **Primary Model (Regression):** Predict the continuous `Final Exam Score` directly from the 10 predictor features.
* **Post-Processing (Rule-Based Binning):** Map the predicted numerical scores directly into categorical letter grades (`A` through `F`) using the standard threshold rules.

By setting `Final Exam Score` as our new regression target, we eliminate data leakage while retaining the ability to generate both continuous score predictions and grade classifications from a single pipeline.

In [12]:
df.isnull().sum()

student_id                      0
gender                          0
study_time_hours                0
attendance_percent              0
sleep_hours                     0
parental_education            102
internet_access                 0
extracurricular_activities      0
part_time_job                   0
previous_grade                  0
final_exam_score                0
final_grade                     0
dtype: int64

As identified during initial inspection, `Parental Education` contains 102 missing values (roughly 10% of the dataset). Handling these during the preprocessing phase—via mode imputation, an "Unknown" category, or predictive encoding—will be essential before feeding the data into our regression models.

In [13]:
df.duplicated().sum()

np.int64(0)

With zero duplicate rows across all 1,000 records, the dataset retains maximum information density for its size. Every entry represents a distinct student profile, ensuring diversity across study habits, demographics, and outcomes—a promising sign for data quality prior to splitting and training.

In [14]:
# Cardinality check
print(f'{"Features":40} Cardinality\tData Type')
for index, feature in enumerate(df.columns[:-2], start=1):
    print(f'{index:2}. {" ".join(feature.split("_")).title():40} {df[feature].nunique()/df.shape[0]:2.2%} \t {str(df[feature].dtype)}')

Features                                 Cardinality	Data Type
 1. Student Id                               100.00% 	 int64
 2. Gender                                   0.20% 	 object
 3. Study Time Hours                         7.00% 	 float64
 4. Attendance Percent                       32.70% 	 float64
 5. Sleep Hours                              6.50% 	 float64
 6. Parental Education                       0.40% 	 object
 7. Internet Access                          0.20% 	 object
 8. Extracurricular Activities               0.20% 	 object
 9. Part Time Job                            0.20% 	 object
10. Previous Grade                           43.40% 	 float64


Evaluating feature cardinality in a single unified view provides a clear picture of dataset diversity across both numerical and categorical variables:

* **High Cardinality (`Student ID`):** Shows 100% unique values (1,000/1,000), confirming it is purely an identifier with zero predictive value that must be removed.
* **Low Numerical Cardinality (`Study Hours`, `Sleep Hours`):** Hovering at just 6–7% unique values, these confirm our variance findings—most students report identical, discrete time allocations, limiting their predictive variance.
* **Moderate Numerical Cardinality (`Attendance Percentage`, `Previous Grade`):** Possess healthy cardinalities of ~33% and ~42%, indicating a rich, continuous spread of values essential for modeling variance in student performance.
* **Low Categorical Cardinality:** Categorical variables like `Gender` (~0.2%), `Internet Access`, and `Part-Time Job` naturally exhibit low cardinality due to their binary or few-level nature, which is expected and ideal for encoding.

In [15]:
df.describe()

,student_id,study_time_hours,attendance_percent,sleep_hours,previous_grade,final_exam_score
count,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000
mean,500.500000,3.570700,85.092300,6.799500,69.740900,83.543500
std,288.819436,1.478559,9.270685,1.203527,12.613425,10.341333
min,1.000000,0.500000,54.800000,3.200000,31.300000,46.800000
25%,250.750000,2.600000,78.800000,5.900000,61.000000,76.075000
50%,500.500000,3.600000,85.200000,6.800000,69.600000,83.800000
75%,750.250000,4.500000,91.900000,7.600000,78.400000,91.525000
max,1000.000000,8.100000,100.000000,10.000000,100.000000,100.000000


Through the `describe()` function, we can evaluate summary statistics, inspect value ranges, and test our initial domain assumptions against the actual data distributions:

* **`Study Hours` vs. Performance:**
  * *Initial Expectation:* From a practical standpoint, one might expect a standard target study workload of roughly 6 hours daily (out of 24) to secure top academic outcomes.
  * *Data Observation:* The data reveals an average of only 3.6 hours per week, with 75% of students studying 4.5 hours or fewer. The maximum of 8 hours represents a distinct, rare edge case rather than the norm.
  * *Analytical Takeaway:* Despite studying an average of just ~3.6 hours (with a minimum of 0.5 hours), students achieve a surprisingly high mean final exam score of 83.5%. It will be critical to examine how strongly study duration actually correlates with top-tier scores, or whether students are simply operating with high baseline efficiency.


* **`Attendance Percentage`:**
  * *Data Observation:* Attendance shows a balanced distribution, ranging from a minimum of 54% to a maximum of 100%, with an average of 85% and a 75th percentile of 91%.
  * *Analytical Takeaway:* The metric scales steadily across interquartile ranges without extreme or erratic outliers, serving as a clean, reliable measure of classroom engagement.


* **`Sleep Hours`:**
  * *Initial Expectation:* Healthy lifestyle standards generally dictate a minimum of 6 hours of sleep per day, with an ideal target of around 8 hours.
  * *Data Observation:* The cohort averages a healthy 7 hours per day. However, extreme cases drop as low as 3.2 hours.
  * *Analytical Takeaway:* Sleeping just 3.2 hours represents a notable lifestyle concern. It will be important to isolate these sleep-deprived individuals to determine if their severely reduced rest directly depresses their final exam outcomes.


* **`Previous Grade`:**
  * *Data Observation:* Past performance averages a solid 70%, spanning a wide range from 31% to a perfect 100% with a consistent interquartile spread.
  * *Analytical Takeaway:* The population generally reflects solid past performance. However, students at the lower bound (31%) lie outside the main normal distribution—representing a key sub-group of struggling students whose trajectory into the final exam warrants targeted analysis.

In [16]:
df.final_grade.value_counts(normalize=True) * 100

final_grade
B    35.4
A    28.4
C    26.1
D     8.9
F     1.2
Name: proportion, dtype: float64

While a perfectly balanced class distribution is often preferred statistically, real-world data rarely works that way—especially in education. A dataset of 1,000 students should naturally exhibit realistic variance across performance tiers rather than artificial symmetry.

Looking at the target grade breakdown, the distribution reflects a remarkably high-performing cohort:

* **Grade B (~35%):** Forms the single largest group. My initial expectation was that an average student body would cluster even more heavily here—around 80% in the middle. Seeing only 35% in Grade B reveals a much wider spread than anticipated.
* **Grade A (~28%):** Represents a surprisingly large high-achieving segment. Nearly a third of the cohort is performing at an elite level.
* **Grade C (~26%):** Accounts for a substantial portion of lower-average performance. While not failing, having over a quarter of the class in this tier highlights a notable group that may need targeted support.
* **Grades D & F (~9-1% each):** Unusually low failure rates. From an institutional health perspective, having under 2% failing or near-failing points to an exceptionally strong educational environment or lenient grading scale.

This distribution challenges the assumption of a tight bell curve centered on average performance. While the vast majority of students are succeeding (A–C), the split is far broader than expected. Statistically, the extreme imbalance in Grades D and F means classification models will face class imbalance issues for low-tier outcomes, but domain-wise, it reflects a thriving student body.

## **Data Analysis**

In [17]:
reg_target = 'final_exam_score'
clf_target = 'final_grade'

In [18]:
fig = px.pie(
    df, 
    names=clf_target, 
    title="Grades Distribution - Donut Chart",
    hole=0.4
).update_traces(textinfo='label+percent')
fig.show()
fig.data

(Pie({
     'domain': {'x': [0.0, 1.0], 'y': [0.0, 1.0]},
     'hole': 0.4,
     'hovertemplate': 'final_grade=%{label}<extra></extra>',
     'labels': array(['A', 'A', 'A', ..., 'C', 'B', 'C'], dtype=object),
     'legendgroup': '',
     'name': '',
     'showlegend': True,
     'textinfo': 'label+percent'
 }),)

Re-examining the target breakdown—with Grade B at 35%, A at 28%, C at 26%, D at 9%, and F at 1%—highlights the analytical value of our modeling pivot:

* **Classification Constraints:** In a purely categorical setup, the severe scarcity of D and F grades (~10% combined) would pose class imbalance challenges, risking poor precision on lower-tier predictions without resampling or custom loss weighting.
* **Regression Advantage:** Because we transitioned to continuous regression, target imbalance on discrete classes is no longer a primary model barrier. The regression target operates on a smooth numerical distribution (`Final Exam Score`), capturing fine-grained variations without suffering from sparse bin artifacts.
* **Pipeline Focus:** Since post-hoc rule-based binning converts predicted scores back into letter grades, our priority shifts to evaluating the regression target's statistical properties—such as skewness, dispersion, and continuous relationships with features—to optimize numerical score accuracy.

In [19]:
fig = px.histogram(
    df,
    x=reg_target,
    color=clf_target,
    title="Final Exam Score Distribution with Grade",
    text_auto=True,
)

fig.update_layout(
    xaxis_title="Score",
    yaxis_title="Frequency"
)

fig.show()
fig.data

(Histogram({
     'alignmentgroup': 'True',
     'bingroup': 'x',
     'hovertemplate': 'final_grade=A<br>final_exam_score=%{x}<br>count=%{y}<extra></extra>',
     'legendgroup': 'A',
     'marker': {'color': '#636efa', 'pattern': {'shape': ''}},
     'name': 'A',
     'offsetgroup': 'A',
     'orientation': 'v',
     'showlegend': True,
     'texttemplate': '%{value}',
     'x': array([100. , 100. ,  97.3, ...,  93.9,  92.7,  99.5]),
     'xaxis': 'x',
     'yaxis': 'y'
 }),
 Histogram({
     'alignmentgroup': 'True',
     'bingroup': 'x',
     'hovertemplate': 'final_grade=B<br>final_exam_score=%{x}<br>count=%{y}<extra></extra>',
     'legendgroup': 'B',
     'marker': {'color': '#EF553B', 'pattern': {'shape': ''}},
     'name': 'B',
     'offsetgroup': 'B',
     'orientation': 'v',
     'showlegend': True,
     'texttemplate': '%{value}',
     'x': array([83.8, 81.6, 88. , ..., 81.9, 80.4, 82.2]),
     'xaxis': 'x',
     'yaxis': 'y'
 }),
 Histogram({
     'alignmentgroup': 'True',


An analysis of the **Final Exam Score Distribution by Grade** histogram reveals key insights into score boundaries, distributional shape, and data artifacts:

* **Strict Bin Boundaries (Rule-Based Cutoffs):** The plot clearly displays hard, non-overlapping score thresholds that define each categorical letter grade along the continuous axis:
* **F (Orange):** Scores below ~60 (with isolated low entries at ~46 and ~53).
* **D (Green):** Scores spanning approximately 60 to 71.
* **C (Purple):** Scores spanning approximately 71 to 81.
* **B (Red):** Scores spanning approximately 81 to 91.
* **A (Blue):** Scores spanning 91 to 100.


* **Distribution Shape & Skewness:** The overall continuous score distribution exhibits a strong **left-skew (negative skewness)**, with the vast majority of student scores concentrated in the 70–100 range. The peak frequency rests in the middle-to-high performance tiers (C, B, and A grades), confirming our earlier finding that lower-tier grades (D and F) represent a tiny tail of the population.
* **Boundary Spike Artifact (Ceiling Effect at 100):** A massive spike occurs in the final bin at **Score 100 (96 students)**. This indicates a classic "ceiling effect" in the dataset—where exam scores are capped at a maximum of 100%, causing a high concentration of top-performing students to pool at the upper boundary.
* **Modeling Takeaway:** When training our regression models, standard algorithms (like linear models) may struggle with the left-skew and the artificial spike at 100. Tree-based models (e.g., XGBoost, LightGBM, Random Forest) will handle these non-linear boundary caps far more effectively without requiring heavy target transformations.

In [20]:
features_to_analyze = list(df.columns[1:-2])

print(f'{"Feature":30}  Data Type')
for idx, name in enumerate(features_to_analyze, start=1):
    print(f'{idx}. {name:30}{df[name].dtype}')

Feature                         Data Type
1. gender                        object
2. study_time_hours              float64
3. attendance_percent            float64
4. sleep_hours                   float64
5. parental_education            object
6. internet_access               object
7. extracurricular_activities    object
8. part_time_job                 object
9. previous_grade                float64


In [21]:
fig = px.pie(
    df, 
    names="gender", 
    title="Gender Distribution - Donut Chart",
    hole=0.4
).update_traces(textinfo='label+percent')

fig.show()
fig.data

(Pie({
     'domain': {'x': [0.0, 1.0], 'y': [0.0, 1.0]},
     'hole': 0.4,
     'hovertemplate': 'gender=%{label}<extra></extra>',
     'labels': array(['Male', 'Female', 'Male', ..., 'Female', 'Female', 'Male'], dtype=object),
     'legendgroup': '',
     'name': '',
     'showlegend': True,
     'textinfo': 'label+percent'
 }),)

The gender distribution in this dataset is exceptionally clean, featuring an almost exact 50/50 split between male and female students.

This balance is ideal for analytical integrity. It ensures that our demographic features carry no inherent representation bias, allowing downstream models to learn fair, unbiased relationships without skewing predictions toward one group.

In [22]:
feature = 'gender'
fig = px.sunburst(
    df, 
    path=[feature, clf_target],
    title=" ".join(feature.split("_")).title() + " and Grade Distribution",
)
fig.show()
print(fig.data)

fig = px.box(
    df,
    x=reg_target,
    color=feature,
    title="Final Exam Scores vs " + " ".join(feature.split("_")).title()
)
fig.show()
fig.data

(Sunburst({
    'branchvalues': 'total',
    'domain': {'x': [0.0, 1.0], 'y': [0.0, 1.0]},
    'hovertemplate': 'labels=%{label}<br>count=%{value}<br>parent=%{parent}<br>id=%{id}<extra></extra>',
    'ids': array(['Female/A', 'Male/A', 'Female/B', 'Male/B', 'Female/C', 'Male/C',
                  'Female/D', 'Male/D', 'Female/F', 'Male/F', 'Female', 'Male'],
                 dtype=object),
    'labels': array(['A', 'A', 'B', 'B', 'C', 'C', 'D', 'D', 'F', 'F', 'Female', 'Male'],
                    dtype=object),
    'name': '',
    'parents': array(['Female', 'Male', 'Female', 'Male', 'Female', 'Male', 'Female', 'Male',
                      'Female', 'Male', '', ''], dtype=object),
    'values': array([140, 144, 186, 168, 134, 127,  44,  45,   6,   6, 510, 490])
}),)


(Box({
     'alignmentgroup': 'True',
     'hovertemplate': 'gender=Male<br>final_exam_score=%{x}<extra></extra>',
     'legendgroup': 'Male',
     'marker': {'color': '#636efa'},
     'name': 'Male',
     'notched': False,
     'offsetgroup': 'Male',
     'orientation': 'h',
     'showlegend': True,
     'x': array([100. ,  97.3,  83.8, ...,  70.7,  99.5,  70.4]),
     'x0': ' ',
     'xaxis': 'x',
     'y0': ' ',
     'yaxis': 'y'
 }),
 Box({
     'alignmentgroup': 'True',
     'hovertemplate': 'gender=Female<br>final_exam_score=%{x}<extra></extra>',
     'legendgroup': 'Female',
     'marker': {'color': '#EF553B'},
     'name': 'Female',
     'notched': False,
     'offsetgroup': 'Female',
     'orientation': 'h',
     'showlegend': True,
     'x': array([100. ,  81.6,  74.3, ...,  79.1,  79.2,  82.2]),
     'x0': ' ',
     'xaxis': 'x',
     'y0': ' ',
     'yaxis': 'y'
 }))

Evaluating final exam scores across gender through both the sunburst chart and box plot confirms that gender carries virtually zero predictive power in this dataset:

* **Categorical Breakdown (Sunburst Chart):** Grade distribution splits (A through F) across males and females are virtually identical. Neither gender dominates the top tiers or disproportionately populates the lower grades.
* **Distributional Spread (Box Plot):**
* *Interquartile Range (IQR):* Both groups share nearly identical middle-50% bands—spanning roughly 76% to 90% for females and 75% to 92% for males. The imperceptible variation in IQR is driven entirely by the slight sample size difference (~51% female vs. ~49% male).
* *Lower Bound:* The lower whisker (lower fence) settles cleanly around 57% for both groups.


* **Identified Outliers:** Extreme low-performing outliers exist across both genders—two females scoring 53% and 54%, and a single male scoring 46%. These rare cases sit far below the main distribution's baseline, reflecting individual risk rather than a demographic pattern.

**Analytical Takeaway:** While societal stereotypes or regional biases often assume girls outperform boys academically, this empirical evidence proves that gender does not affect final exam scores. It functions as a neutral feature with near-zero signal for our regression model.

In [23]:
fig = px.pie(
    df, 
    names="parental_education", 
    title="Parental Education Distribution - Donut Chart",
    hole=0.4
).update_traces(textinfo='label+percent')
fig.show()
fig.data

(Pie({
     'domain': {'x': [0.0, 1.0], 'y': [0.0, 1.0]},
     'hole': 0.4,
     'hovertemplate': 'parental_education=%{label}<extra></extra>',
     'labels': array(['Bachelors', 'High School', 'Bachelors', ..., 'High School',
                      'High School', 'Masters'], dtype=object),
     'legendgroup': '',
     'name': '',
     'showlegend': True,
     'textinfo': 'label+percent'
 }),)

An analysis of the `Parental Education` distribution reveals notable socio-demographic patterns alongside a critical decision point for handling missing data:

* **High School (35%) & Bachelor's (30%):**
  * *Domain Expectation:* Modern expectations often assume a Bachelor's degree is the standard baseline across households regardless of field.
  * *Data Observation:* High school remains the single largest category at 35%, outpacing Bachelor's degrees (30%). It will be compelling to observe whether this lower parental baseline negatively impacts student exam outcomes or if student effort overrides household academic history.
 
* **Master's (18%) & PhD (5%):** Higher degrees taper off as expected, reflecting natural drop-offs in advanced post-graduate enrollment.
  * **Missing Data / Nulls (~10%):**
  * *Data Interpretation:* The ~10% missing entries present a fundamental question: are these randomly dropped data points (Missing Completely at Random), or did parents intentionally withhold their education level due to privacy or social stigma?
  * *Preprocessing Strategy:* Imputing 102 rows with the mode (*High School*) risks artificially inflating the dominant class and distorting feature relationships. Given the 10% volume, treating `Missing` as an explicit, distinct category ("Not Disclosed") preserves potential signal without introducing arbitrary bias before modeling.

In [24]:
# Filling parental_education's nan values with 'Not Disclosed'
df['parental_education'] = df.parental_education.fillna('Not Disclosed')

In [25]:
feature = 'parental_education'
fig = px.sunburst(
    df, 
    path=[feature, clf_target],
    title=" ".join(feature.split("_")).title() + " and Grade Distribution",
)
fig.show()
print(fig.data)

fig = px.box(
    df,
    x=reg_target,
    color=feature,
    title="Final Exam Scores vs " + " ".join(feature.split("_")).title()
)
fig.show()
fig.data

(Sunburst({
    'branchvalues': 'total',
    'domain': {'x': [0.0, 1.0], 'y': [0.0, 1.0]},
    'hovertemplate': 'labels=%{label}<br>count=%{value}<br>parent=%{parent}<br>id=%{id}<extra></extra>',
    'ids': array(['Bachelors/A', 'High School/A', 'Masters/A', 'Not Disclosed/A', 'PhD/A',
                  'Bachelors/B', 'High School/B', 'Masters/B', 'Not Disclosed/B', 'PhD/B',
                  'Bachelors/C', 'High School/C', 'Masters/C', 'Not Disclosed/C', 'PhD/C',
                  'Bachelors/D', 'High School/D', 'Masters/D', 'Not Disclosed/D', 'PhD/D',
                  'Bachelors/F', 'High School/F', 'Masters/F', 'Not Disclosed/F', 'PhD/F',
                  'Bachelors', 'High School', 'Masters', 'Not Disclosed', 'PhD'],
                 dtype=object),
    'labels': array(['A', 'A', 'A', 'A', 'A', 'B', 'B', 'B', 'B', 'B', 'C', 'C', 'C', 'C',
                     'C', 'D', 'D', 'D', 'D', 'D', 'F', 'F', 'F', 'F', 'F', 'Bachelors',
                     'High School', 'Masters', 'Not Dis

(Box({
     'alignmentgroup': 'True',
     'hovertemplate': 'parental_education=Bachelors<br>final_exam_score=%{x}<extra></extra>',
     'legendgroup': 'Bachelors',
     'marker': {'color': '#636efa'},
     'name': 'Bachelors',
     'notched': False,
     'offsetgroup': 'Bachelors',
     'orientation': 'h',
     'showlegend': True,
     'x': array([100. ,  97.3,  68.3, ...,  75.4,  80.4,  79.1]),
     'x0': ' ',
     'xaxis': 'x',
     'y0': ' ',
     'yaxis': 'y'
 }),
 Box({
     'alignmentgroup': 'True',
     'hovertemplate': 'parental_education=High School<br>final_exam_score=%{x}<extra></extra>',
     'legendgroup': 'High School',
     'marker': {'color': '#EF553B'},
     'name': 'High School',
     'notched': False,
     'offsetgroup': 'High School',
     'orientation': 'h',
     'showlegend': True,
     'x': array([100. ,  81.6,  69.6, ...,  78.8,  79.2,  82.2]),
     'x0': ' ',
     'xaxis': 'x',
     'y0': ' ',
     'yaxis': 'y'
 }),
 Box({
     'alignmentgroup': 'True',
     '

Analyzing the relationship between `Parental Education` and final performance yields surprising insights that challenge typical socioeconomic assumptions:

* **Overlap Across Primary Categories:**
* *Domain Expectation:* Higher parental education (e.g., Master's or PhD) is often assumed to correlate with higher exam scores, driven by superior academic guidance, technological access, and household resources.
* *Data Observation:* For High School, Bachelor's, and Master's groups, the interquartile ranges (IQRs) overlap almost entirely. While minor differences exist in min/max bounds, the dominant grade ordering remains consistently **B → A → C → D → F**. This suggests parental degree level alone does not dictate student outcomes.


* **Anomalous Distribution in PhD Households:**
* *Data Observation:* PhD households break the standard grade pattern, shifting to a non-standard order of **A → C → B → D**.
* *Analytical Takeaway:* While Grade A is the single largest tier for children of PhD holders, the second largest group falls to Grade C rather than B. Students in this subset appear bimodal—either performing at an elite level or dropping to a below-average tier.


* **Sample Size & Generalization Constraints:** Because PhD parents represent only \~5% of the total dataset (\~50 records), this subset divergence may be a small-sample artifact rather than a universal trend. Within the boundaries of this 1,000-row sample, `Parental Education` provides limited predictive separation due to the heavy overlap among the larger tiers.

In [26]:
fig = px.pie(
    df, 
    names="internet_access", 
    title="Internet Access Distribution - Donut Chart",
    hole=0.4
).update_traces(textinfo='label+percent')
fig.show()
fig.data

(Pie({
     'domain': {'x': [0.0, 1.0], 'y': [0.0, 1.0]},
     'hole': 0.4,
     'hovertemplate': 'internet_access=%{label}<extra></extra>',
     'labels': array(['Yes', 'Yes', 'Yes', ..., 'Yes', 'No', 'Yes'], dtype=object),
     'legendgroup': '',
     'name': '',
     'showlegend': True,
     'textinfo': 'label+percent'
 }),)

Analyzing the distribution of `Internet Access` highlights both expected real-world trends and important analytical limitations:

* **Data Breakdown:** Approximately 85% of students in the dataset report having home internet access, while only 14% do not. This heavy lean aligns directly with modern expectations, where internet availability is the default for the vast majority of households.
* **Analytical Limitation (Indirect Signal):** As noted earlier, binary internet access is an indirect predictor of academic success. Simply having connectivity reveals nothing about *how* a student utilizes it—whether for online coursework, research, gaming, or social media. A far more informative metric would capture daily screen time allocation or internet activity type.
* **Exploratory Value:** From a data analytics perspective, while the relationship between raw access and final performance is non-direct, evaluating this variable against target exam scores will be intriguing. It allows us to test whether lacking basic connectivity acts as an absolute barrier to performance or if motivated students overcome it entirely.

In [27]:
feature = 'internet_access'
fig = px.sunburst(
    df, 
    path=[feature, clf_target],
    title=" ".join(feature.split("_")).title() + " and Grade Distribution",
)
fig.show()
fig.data

(Sunburst({
     'branchvalues': 'total',
     'domain': {'x': [0.0, 1.0], 'y': [0.0, 1.0]},
     'hovertemplate': 'labels=%{label}<br>count=%{value}<br>parent=%{parent}<br>id=%{id}<extra></extra>',
     'ids': array(['No/A', 'Yes/A', 'No/B', 'Yes/B', 'No/C', 'Yes/C', 'No/D', 'Yes/D',
                   'No/F', 'Yes/F', 'No', 'Yes'], dtype=object),
     'labels': array(['A', 'A', 'B', 'B', 'C', 'C', 'D', 'D', 'F', 'F', 'No', 'Yes'],
                     dtype=object),
     'name': '',
     'parents': array(['No', 'Yes', 'No', 'Yes', 'No', 'Yes', 'No', 'Yes', 'No', 'Yes', '', ''],
                      dtype=object),
     'values': array([ 28, 256,  35, 319,  57, 204,  20,  69,   6,   6, 146, 854])
 }),)

In [28]:
feature = 'internet_access'
fig = px.sunburst(
    df, 
    path=[feature, clf_target],
    title=" ".join(feature.split("_")).title() + " and Grade Distribution",
)
fig.show()
print(fig.data)

fig = px.box(
    df,
    x=reg_target,
    color=feature,
    title="Final Exam Scores vs " + " ".join(feature.split("_")).title()
)
fig.show()
fig.data

(Sunburst({
    'branchvalues': 'total',
    'domain': {'x': [0.0, 1.0], 'y': [0.0, 1.0]},
    'hovertemplate': 'labels=%{label}<br>count=%{value}<br>parent=%{parent}<br>id=%{id}<extra></extra>',
    'ids': array(['No/A', 'Yes/A', 'No/B', 'Yes/B', 'No/C', 'Yes/C', 'No/D', 'Yes/D',
                  'No/F', 'Yes/F', 'No', 'Yes'], dtype=object),
    'labels': array(['A', 'A', 'B', 'B', 'C', 'C', 'D', 'D', 'F', 'F', 'No', 'Yes'],
                    dtype=object),
    'name': '',
    'parents': array(['No', 'Yes', 'No', 'Yes', 'No', 'Yes', 'No', 'Yes', 'No', 'Yes', '', ''],
                     dtype=object),
    'values': array([ 28, 256,  35, 319,  57, 204,  20,  69,   6,   6, 146, 854])
}),)


(Box({
     'alignmentgroup': 'True',
     'hovertemplate': 'internet_access=Yes<br>final_exam_score=%{x}<extra></extra>',
     'legendgroup': 'Yes',
     'marker': {'color': '#636efa'},
     'name': 'Yes',
     'notched': False,
     'offsetgroup': 'Yes',
     'orientation': 'h',
     'showlegend': True,
     'x': array([100. , 100. ,  97.3, ...,  70.7,  79.2,  70.4]),
     'x0': ' ',
     'xaxis': 'x',
     'y0': ' ',
     'yaxis': 'y'
 }),
 Box({
     'alignmentgroup': 'True',
     'hovertemplate': 'internet_access=No<br>final_exam_score=%{x}<extra></extra>',
     'legendgroup': 'No',
     'marker': {'color': '#EF553B'},
     'name': 'No',
     'notched': False,
     'offsetgroup': 'No',
     'orientation': 'h',
     'showlegend': True,
     'x': array([ 93.3,  89.7,  92.8,  73.4,  71.2,  97.4,  93.3,  70.8,  61.8,  88.6,
                  81.8,  99.8,  71.1,  79.2,  73.5,  64.4,  88.5,  79.2,  89.5,  75.5,
                  91.4,  72.7,  94.5,  59.3,  77.3,  59.7,  75.7,  75.1,  78

Analyzing `Internet Access` against final performance yields our first statistically meaningful finding, uncovering a clear positive relationship:

* **Distributional Shift (Box Plot Insights):**
  * *Interquartile Range (IQR):* Students without internet access exhibit a lower performance band (72% to 88%), whereas those with internet access shift upward (77% to 92%).
  * *Domain Impact:* While raw binary access cannot tell us *how* connectivity is used, the baseline elevation indicates that digital access generally supports academic productivity—whether through study resources, peer collaboration, or educational media—rather than acting purely as a distraction.
  * *Outliers:* Despite this positive trend, two extreme low-performing outliers (53% and 46%) possess internet access, proving that connectivity alone does not guarantee success.


* **Categorical Hierarchy (Sunburst Chart Insights):**
  * *With Internet Access:* Follows the high-performing **B → A → C → D** rank order, where the majority of students achieve average (B) to elite (A) grades.
  * *Without Internet Access:* Shifts to a **C → B → A → D** rank order. Grade C becomes the primary category, indicating that lacking internet access pushes a larger proportion of students into below-average performance tiers.



**Analytical Takeaway:** Contrary to the assumption that open internet access primarily leads to unproductivity, empirical evidence demonstrates that connected students systematically outperform their non-connected peers across both median score and overall grade distribution.

In [29]:
fig = px.pie(
    df, 
    names="extracurricular_activities", 
    title="Extracurricular Activities Distribution - Donut Chart",
    hole=0.4
).update_traces(textinfo='label+percent')
fig.show()
fig.data

(Pie({
     'domain': {'x': [0.0, 1.0], 'y': [0.0, 1.0]},
     'hole': 0.4,
     'hovertemplate': 'extracurricular_activities=%{label}<extra></extra>',
     'labels': array(['Yes', 'Yes', 'No', ..., 'Yes', 'No', 'No'], dtype=object),
     'legendgroup': '',
     'name': '',
     'showlegend': True,
     'textinfo': 'label+percent'
 }),)

An analysis of `Extracurricular Activities` reveals a balanced distribution across the student body, opening up two competing hypotheses regarding its impact:

* **Data Breakdown:** Approximately 57% of students participate in extracurricular activities, while 42% do not. This near-even split provides a clean, well-balanced sample to test against performance without severe class imbalance.
* **Competing Domain Hypotheses:**
  * *The Time-Tradeoff View:* Non-participating students may allocate those extra hours directly toward studying, potentially yielding higher academic scores.
  * *The Refreshment View:* Participating students may use extracurriculars as a productive mental outlet to reduce burnout. A refreshed mindset can enhance focus during study sessions, potentially neutralizing or even boosting overall efficiency.



* **Analytical Value:** Without strong prior assumptions leaning toward either outcome, this balanced distribution provides an ideal setup for EDA. Cross-tabulating participation against final exam scores will isolate whether extracurricular engagement serves as a distraction or a performance enhancer.

In [30]:
feature = 'extracurricular_activities'
fig = px.sunburst(
    df, 
    path=[feature, clf_target],
    title=" ".join(feature.split("_")).title() + " and Grade Distribution",
)
fig.show()
print(fig.data)

fig = px.box(
    df,
    x=reg_target,
    color=feature,
    title="Final Exam Scores vs " + " ".join(feature.split("_")).title()
)
fig.show()
fig.data

(Sunburst({
    'branchvalues': 'total',
    'domain': {'x': [0.0, 1.0], 'y': [0.0, 1.0]},
    'hovertemplate': 'labels=%{label}<br>count=%{value}<br>parent=%{parent}<br>id=%{id}<extra></extra>',
    'ids': array(['No/A', 'Yes/A', 'No/B', 'Yes/B', 'No/C', 'Yes/C', 'No/D', 'Yes/D',
                  'No/F', 'Yes/F', 'No', 'Yes'], dtype=object),
    'labels': array(['A', 'A', 'B', 'B', 'C', 'C', 'D', 'D', 'F', 'F', 'No', 'Yes'],
                    dtype=object),
    'name': '',
    'parents': array(['No', 'Yes', 'No', 'Yes', 'No', 'Yes', 'No', 'Yes', 'No', 'Yes', '', ''],
                     dtype=object),
    'values': array([113, 171, 156, 198, 119, 142,  37,  52,   3,   9, 428, 572])
}),)


(Box({
     'alignmentgroup': 'True',
     'hovertemplate': 'extracurricular_activities=Yes<br>final_exam_score=%{x}<extra></extra>',
     'legendgroup': 'Yes',
     'marker': {'color': '#636efa'},
     'name': 'Yes',
     'notched': False,
     'offsetgroup': 'Yes',
     'orientation': 'h',
     'showlegend': True,
     'x': array([100. , 100. ,  83.8, ...,  70.7,  99.5,  79.2]),
     'x0': ' ',
     'xaxis': 'x',
     'y0': ' ',
     'yaxis': 'y'
 }),
 Box({
     'alignmentgroup': 'True',
     'hovertemplate': 'extracurricular_activities=No<br>final_exam_score=%{x}<extra></extra>',
     'legendgroup': 'No',
     'marker': {'color': '#EF553B'},
     'name': 'No',
     'notched': False,
     'offsetgroup': 'No',
     'orientation': 'h',
     'showlegend': True,
     'x': array([97.3, 68.3, 81.6, ..., 92.7, 82.2, 70.4]),
     'x0': ' ',
     'xaxis': 'x',
     'y0': ' ',
     'yaxis': 'y'
 }))

Evaluating `Extracurricular Activities` against final exam scores reveals subtle yet meaningful structural differences between continuous ranges and categorical grade hierarchies:

* **Continuous Score Distribution (Box Plot Insights):**
  * *Interquartile Range (IQR):* The continuous score bands for participants and non-participants show extensive overlap, maintaining nearly identical middle-50% ranges. Min-max spreads remain broad for both groups, indicating that participation alone does not drastically alter the median continuous exam score.


* **Categorical Grade Hierarchies (Sunburst Chart Insights):**
  * *Participating Students:* Follow the high-performing **B → A → C → D** trajectory. These students cluster heavily in the average (B) and elite (A) tiers.
  * *Non-Participating Students:* Shift to a **B → C → A → D** trajectory. While Grade B remains the primary group, Grade C overtakes Grade A as the second-largest category.


* **Domain & Institutional Strategy:** Non-participating students carry a visibly higher probability of sliding into below-average performance (Grade C) rather than reaching top-tier outcomes (Grade A).

**Analytical Takeaway:** While extracurricular engagement does not guarantee dramatically higher numeric scores, it appears to act as a protective buffer against below-average outcomes. From an institutional perspective, encouraging non-active students to join structured activities could provide the mental refreshment needed to elevate at-risk students from C-tier to B-tier performance.

In [31]:
fig = px.pie(
    df, 
    names="part_time_job", 
    title="Part Time Job Distribution - Donut Chart",
    hole=0.4
).update_traces(textinfo='label+percent')
fig.show()
fig.data

(Pie({
     'domain': {'x': [0.0, 1.0], 'y': [0.0, 1.0]},
     'hole': 0.4,
     'hovertemplate': 'part_time_job=%{label}<extra></extra>',
     'labels': array(['No', 'Yes', 'Yes', ..., 'Yes', 'Yes', 'No'], dtype=object),
     'legendgroup': '',
     'name': '',
     'showlegend': True,
     'textinfo': 'label+percent'
 }),)

Analyzing `Part-Time Job` reveals an expected real-world imbalance that sets up a compelling hypothesis for performance modeling:

* **Data Breakdown:** Approximately 31% of students hold a part-time job, while the remaining ~68% do not.
* **Domain Context:** This skew aligns well with real-world expectations. Full-time students typically prioritize their studies, making non-working students the natural majority in an academic cohort.
* **Analytical Impact:** Evaluating this binary feature against final exam scores will isolate the trade-off between financial/work commitments and academic yield. It allows us to test whether working a part-time job imposes time constraints that depress performance, or if working students demonstrate superior time-management skills that offset the reduced study hours.

In [32]:
feature = 'part_time_job'
fig = px.sunburst(
    df, 
    path=[feature, clf_target],
    title=" ".join(feature.split("_")).title() + " and Grade Distribution",
)
fig.show()
print(fig.data)

fig = px.box(
    df,
    x=reg_target,
    color=feature,
    title="Final Exam Scores vs " + " ".join(feature.split("_")).title()
)
fig.show()
fig.data

(Sunburst({
    'branchvalues': 'total',
    'domain': {'x': [0.0, 1.0], 'y': [0.0, 1.0]},
    'hovertemplate': 'labels=%{label}<br>count=%{value}<br>parent=%{parent}<br>id=%{id}<extra></extra>',
    'ids': array(['No/A', 'Yes/A', 'No/B', 'Yes/B', 'No/C', 'Yes/C', 'No/D', 'Yes/D',
                  'No/F', 'Yes/F', 'No', 'Yes'], dtype=object),
    'labels': array(['A', 'A', 'B', 'B', 'C', 'C', 'D', 'D', 'F', 'F', 'No', 'Yes'],
                    dtype=object),
    'name': '',
    'parents': array(['No', 'Yes', 'No', 'Yes', 'No', 'Yes', 'No', 'Yes', 'No', 'Yes', '', ''],
                     dtype=object),
    'values': array([213,  71, 245, 109, 180,  81,  41,  48,   5,   7, 684, 316])
}),)


(Box({
     'alignmentgroup': 'True',
     'hovertemplate': 'part_time_job=No<br>final_exam_score=%{x}<extra></extra>',
     'legendgroup': 'No',
     'marker': {'color': '#636efa'},
     'name': 'No',
     'notched': False,
     'offsetgroup': 'No',
     'orientation': 'h',
     'showlegend': True,
     'x': array([100. ,  83.8,  81.6, ...,  70.7,  99.5,  70.4]),
     'x0': ' ',
     'xaxis': 'x',
     'y0': ' ',
     'yaxis': 'y'
 }),
 Box({
     'alignmentgroup': 'True',
     'hovertemplate': 'part_time_job=Yes<br>final_exam_score=%{x}<extra></extra>',
     'legendgroup': 'Yes',
     'marker': {'color': '#EF553B'},
     'name': 'Yes',
     'notched': False,
     'offsetgroup': 'Yes',
     'orientation': 'h',
     'showlegend': True,
     'x': array([100. ,  97.3,  68.3, ...,  93.9,  79.2,  82.2]),
     'x0': ' ',
     'xaxis': 'x',
     'y0': ' ',
     'yaxis': 'y'
 }))

Analyzing `Part-Time Job` against final exam scores reveals a clear, quantifiable trade-off between employment commitments and academic performance:

* **Distributional Shift (Box Plot Insights):**
  * *Interquartile Range (IQR):* Non-working students exhibit a elevated performance band ($Q_1 = 77\%$, $Q_3 = 92\%$), whereas working students shift lower ($Q_1 = 73\%$, $Q_3 = 89\%$).
  * *Median Difference:* Non-working students achieve a higher median score of 84%, compared to 82% for those with part-time jobs.
  * *Domain Reality:* While medians of 82% and 84% both represent strong absolute performance, the systematic downward shift highlights the time-budget constraint: working hours directly compete with dedicated study time.


* **Categorical Hierarchy (Sunburst Chart Insights):**
  * *No Part-Time Job:* Follows the high-performing **B → A → C → D** rank order, maximizing the probability of achieving top-tier grades (A and B).
  * *With Part-Time Job:* Shifts to a **B → C → A → D** rank order, where Grade C overtakes Grade A as the second most common outcome.



**Analytical Takeaway:** While taking on a part-time job does not cause students to fail outright, it systematically compresses peak performance. Employed students are significantly more likely to settle into average (B) or below-average (C) tiers rather than ascending to elite (A) status.

In [33]:
feature = num_features[1]

fig = px.histogram(
    df,
    x=feature,
    text_auto=True,
    title=" ".join(feature.split('_')).title() + " Histogram Distribution",
)
fig.show()
fig.data

(Histogram({
     'alignmentgroup': 'True',
     'bingroup': 'x',
     'hovertemplate': 'study_time_hours=%{x}<br>count=%{y}<extra></extra>',
     'legendgroup': '',
     'marker': {'color': '#636efa', 'pattern': {'shape': ''}},
     'name': '',
     'offsetgroup': '',
     'orientation': 'v',
     'showlegend': False,
     'texttemplate': '%{value}',
     'x': array([4. , 6.3, 4.9, ..., 2.6, 4.6, 3.9]),
     'xaxis': 'x',
     'yaxis': 'y'
 }),)

An analysis of `Study Hours` reveals an approximately bell-shaped distribution alongside specific behavioral spikes and drop-offs:

* **Central Tendency & Peak Volume:** The distribution behaves like a near-Gaussian bell curve, peaking in the 3.4–4.5 hour range (reaching up to 66 students per bin). Studying 3.5 to 4 hours appears to be the sweet spot for the typical student in this dataset.
* **Low-Effort Spike:** An unexpected spike appears at the ultra-low end (0.4–0.5 hours), where 23 students report under 30 minutes of daily study time—far outnumbering adjacent bins (e.g., 0.6–0.7 hours). Isolating these low-effort students will help verify whether high efficiency or low engagement drives their final scores.
* **The High-Effort Drop-off:** Beyond 4.5 hours, participation drops sharply (from ~49 students at 4.5 hours down to ~29 students at 5.3 hours), tapering off steadily into a light right-tail. A single high-effort outlier reaches 8.0–8.1 hours of study time.
* **Preprocessing Implications:** Because the overall distribution exhibits minimal right-skewness, applying non-linear log transformations is unnecessary. Standard scaling or tree-friendly continuous binning will adequately handle this feature for regression modeling.

In [34]:
fig = px.box(
    df,
    x=feature,
    color=clf_target,
    title=" ".join(feature.split('_')).title() + " vs "  + " ".join(clf_target.split('_')).title() + " (Box Plot)",
)
fig.show()
print(fig.data)

fig = px.scatter(
    df,
    x=feature,
    y=reg_target,
    trendline='ols',
    marginal_x='histogram',
    marginal_y='histogram',
)
fig.show()
fig.data

(Box({
    'alignmentgroup': 'True',
    'hovertemplate': 'final_grade=A<br>study_time_hours=%{x}<extra></extra>',
    'legendgroup': 'A',
    'marker': {'color': '#636efa'},
    'name': 'A',
    'notched': False,
    'offsetgroup': 'A',
    'orientation': 'h',
    'showlegend': True,
    'x': array([4. , 6.3, 4.9, ..., 3.2, 6.2, 6.7]),
    'x0': ' ',
    'xaxis': 'x',
    'y0': ' ',
    'yaxis': 'y'
}), Box({
    'alignmentgroup': 'True',
    'hovertemplate': 'final_grade=B<br>study_time_hours=%{x}<extra></extra>',
    'legendgroup': 'B',
    'marker': {'color': '#EF553B'},
    'name': 'B',
    'notched': False,
    'offsetgroup': 'B',
    'orientation': 'h',
    'showlegend': True,
    'x': array([2.6, 4.2, 6.2, ..., 2.5, 5.1, 4.6]),
    'x0': ' ',
    'xaxis': 'x',
    'y0': ' ',
    'yaxis': 'y'
}), Box({
    'alignmentgroup': 'True',
    'hovertemplate': 'final_grade=D<br>study_time_hours=%{x}<extra></extra>',
    'legendgroup': 'D',
    'marker': {'color': '#00cc96'},
    'name':

(Scatter({
     'hovertemplate': 'study_time_hours=%{x}<br>final_exam_score=%{y}<extra></extra>',
     'legendgroup': '',
     'marker': {'color': '#636efa', 'symbol': 'circle'},
     'mode': 'markers',
     'name': '',
     'orientation': 'v',
     'showlegend': False,
     'x': array([4. , 6.3, 4.9, ..., 2.6, 4.6, 3.9]),
     'xaxis': 'x',
     'y': array([100. , 100. ,  97.3, ...,  79.2,  82.2,  70.4]),
     'yaxis': 'y'
 }),
 Histogram({
     'alignmentgroup': 'True',
     'bingroup': 'x',
     'hovertemplate': 'study_time_hours=%{x}<br>count=%{y}<extra></extra>',
     'legendgroup': '',
     'marker': {'color': '#636efa'},
     'name': '',
     'offsetgroup': '',
     'opacity': 0.5,
     'showlegend': False,
     'x': array([4. , 6.3, 4.9, ..., 2.6, 4.6, 3.9]),
     'xaxis': 'x3',
     'yaxis': 'y3'
 }),
 Histogram({
     'alignmentgroup': 'True',
     'bingroup': 'y',
     'hovertemplate': 'final_exam_score=%{y}<br>count=%{x}<extra></extra>',
     'legendgroup': '',
     'marker

In [35]:
px.scatter(
    df,
    x=feature,
    y=reg_target,
    trendline='ols',
    marginal_x='histogram',
    marginal_y='histogram',
).data

(Scatter({
     'hovertemplate': 'study_time_hours=%{x}<br>final_exam_score=%{y}<extra></extra>',
     'legendgroup': '',
     'marker': {'color': '#636efa', 'symbol': 'circle'},
     'mode': 'markers',
     'name': '',
     'orientation': 'v',
     'showlegend': False,
     'x': array([4. , 6.3, 4.9, ..., 2.6, 4.6, 3.9]),
     'xaxis': 'x',
     'y': array([100. , 100. ,  97.3, ...,  79.2,  82.2,  70.4]),
     'yaxis': 'y'
 }),
 Histogram({
     'alignmentgroup': 'True',
     'bingroup': 'x',
     'hovertemplate': 'study_time_hours=%{x}<br>count=%{y}<extra></extra>',
     'legendgroup': '',
     'marker': {'color': '#636efa'},
     'name': '',
     'offsetgroup': '',
     'opacity': 0.5,
     'showlegend': False,
     'x': array([4. , 6.3, 4.9, ..., 2.6, 4.6, 3.9]),
     'xaxis': 'x3',
     'yaxis': 'y3'
 }),
 Histogram({
     'alignmentgroup': 'True',
     'bingroup': 'y',
     'hovertemplate': 'final_exam_score=%{y}<br>count=%{x}<extra></extra>',
     'legendgroup': '',
     'marker

Analyzing the relationship between `Study Hours` and `Final Exam Score` across both regression and categorical perspectives highlights key strengths, limitations, and non-linearities:

* **Scatter Plot & Trend Line Evaluation:**
  * *Variance Explained:* An $R^2$ of 0.32 indicates that continuous `Study Hours` explains ~32% of the variance in `Final Exam Score`. While a solid baseline signal, linear modeling falls short of capturing the full picture.
  * *Low-Effort Variance & Ceiling Artifacts:* Students studying under 30 minutes daily exhibit immense score dispersion—ranging from a low of 54% to a high of 88%. This wide spread, paired with the artificial score cap at 100%, introduces severe heteroskedasticity that suppresses linear correlation.


* **Box Plot Insights (Non-Linear Categorical Trends):**
  * *Grade F (1.3–2.3 hrs IQR):* Low study hours strongly cluster within failing grades. However, broad min-max whiskers create heavy overlap across groups, including a single anomaly where a student studied ~4 hours but still received an F.
  * *Grade A (3.8–5.6 hrs IQR):* Top-performing students concentrate heavily in higher study brackets. Notable edge cases—such as a student receiving an A with just 1 hour of study, or students studying 7.2–7.3 hours settling for a B—vividly demonstrate that study efficiency and prior knowledge moderate total effort.
  * *Non-Linear Grade C vs. D Inversion:* While Grade D aligns with lower study hours, Grade C exhibits a counterintuitive upward shift in study time. This non-monotonic behavior explains why linear fits underperform.



**Analytical Takeaway:** Study hours provide a primary anchor for predicting academic success, but the relationship is distinctly non-linear and noise-heavy. Tree-based algorithms or non-linear models will far better leverage these step-wise grade thresholds and efficiency edge cases than standard linear regression.

In [36]:
feature = num_features[2]

fig = px.histogram(
    df,
    x=feature,
    text_auto=True,
    title=" ".join(feature.split('_')).title() + " Histogram Distribution",
)
fig.show()
fig.data

(Histogram({
     'alignmentgroup': 'True',
     'bingroup': 'x',
     'hovertemplate': 'attendance_percent=%{x}<br>count=%{y}<extra></extra>',
     'legendgroup': '',
     'marker': {'color': '#636efa', 'pattern': {'shape': ''}},
     'name': '',
     'offsetgroup': '',
     'orientation': 'v',
     'showlegend': False,
     'texttemplate': '%{value}',
     'x': array([ 98. , 100. ,  85.3, ...,  84.5,  85.3,  77.4]),
     'xaxis': 'x',
     'yaxis': 'y'
 }),)

An analysis of `Attendance Percentage` reveals a heavily left-skewed, multi-modal distribution driven by high student engagement:

* **Distribution Shape & Key Peaks:** The overall distribution displays strong **negative skewness**, with student density concentrated heavily at the upper end. Three distinct peaks emerge across the profile:
  * *Primary Peak:* Counts reach 85 students in the 83–85% range.
  * *Secondary Peak:* Counts reach 84 students in the 89–91% range.
  * *Boundary Spike:* A massive sharp jump appears at the upper limit (99–100%), leaping from 35 students in the adjacent 97–98% bin straight up to 83 students.


* **Left-Tail Dispersion:** A thin, elongated left tail spans from 53% to 65% attendance, representing a small minority of disengaged or high-absenteeism students.
* **Transformation Strategy:** While a logarithmic transformation could technically compress the left-skew, it risks obscuring these distinct multi-modal peaks and the upper boundary spike. Preserving the raw continuous scale maintains the true structural signal of student behavior, which tree-based models can natively exploit without distortion.

In [37]:
fig = px.box(
    df,
    x=feature,
    color=clf_target,
    title=" ".join(feature.split('_')).title() + " vs "  + " ".join(clf_target.split('_')).title() + " (Box Plot)",
)
fig.show()
print(fig.data)

fig = px.scatter(
    df,
    x=feature,
    y=reg_target,
    trendline='ols',
    marginal_x='histogram',
    marginal_y='histogram',
)
fig.show()
fig.data

(Box({
    'alignmentgroup': 'True',
    'hovertemplate': 'final_grade=A<br>attendance_percent=%{x}<extra></extra>',
    'legendgroup': 'A',
    'marker': {'color': '#636efa'},
    'name': 'A',
    'notched': False,
    'offsetgroup': 'A',
    'orientation': 'h',
    'showlegend': True,
    'x': array([ 98. , 100. ,  85.3, ...,  99.5,  87.6,  88.4]),
    'x0': ' ',
    'xaxis': 'x',
    'y0': ' ',
    'yaxis': 'y'
}), Box({
    'alignmentgroup': 'True',
    'hovertemplate': 'final_grade=B<br>attendance_percent=%{x}<extra></extra>',
    'legendgroup': 'B',
    'marker': {'color': '#EF553B'},
    'name': 'B',
    'notched': False,
    'offsetgroup': 'B',
    'orientation': 'h',
    'showlegend': True,
    'x': array([77.5, 78.2, 86.4, ..., 81.4, 71.7, 85.3]),
    'x0': ' ',
    'xaxis': 'x',
    'y0': ' ',
    'yaxis': 'y'
}), Box({
    'alignmentgroup': 'True',
    'hovertemplate': 'final_grade=D<br>attendance_percent=%{x}<extra></extra>',
    'legendgroup': 'D',
    'marker': {'color':

(Scatter({
     'hovertemplate': 'attendance_percent=%{x}<br>final_exam_score=%{y}<extra></extra>',
     'legendgroup': '',
     'marker': {'color': '#636efa', 'symbol': 'circle'},
     'mode': 'markers',
     'name': '',
     'orientation': 'v',
     'showlegend': False,
     'x': array([ 98. , 100. ,  85.3, ...,  84.5,  85.3,  77.4]),
     'xaxis': 'x',
     'y': array([100. , 100. ,  97.3, ...,  79.2,  82.2,  70.4]),
     'yaxis': 'y'
 }),
 Histogram({
     'alignmentgroup': 'True',
     'bingroup': 'x',
     'hovertemplate': 'attendance_percent=%{x}<br>count=%{y}<extra></extra>',
     'legendgroup': '',
     'marker': {'color': '#636efa'},
     'name': '',
     'offsetgroup': '',
     'opacity': 0.5,
     'showlegend': False,
     'x': array([ 98. , 100. ,  85.3, ...,  84.5,  85.3,  77.4]),
     'xaxis': 'x3',
     'yaxis': 'y3'
 }),
 Histogram({
     'alignmentgroup': 'True',
     'bingroup': 'y',
     'hovertemplate': 'final_exam_score=%{y}<br>count=%{x}<extra></extra>',
     'le

Analyzing `Attendance Percentage` against final academic performance through both categorical box plots and continuous regression metrics reveals stark contrast between extreme grade boundaries and middle-tier overlap:

* **Box Plot Insights (Separation at the Extremes):**
  * *Boundary Contrast:* The extreme performance tiers show clear separation. Failing students (Grade F) exhibit a lower attendance IQR (70.2%–74.2%), whereas top-tier students (Grade A) shift significantly higher (82.8%–94.5%). Higher attendance directly correlates with elite academic positioning.
  * *Middle-Tier Overlap (B, C, and D):* Mid-level grades exhibit heavy interquartile overlap. Grade C (77%–90%) and Grade B (78%–92%) share nearly identical core distributions.
  * *High-Variance Outliers in Grade D:* Grade D demonstrates the widest overall min-max spread (55.6% to 100%), with an IQR of 73%–87.9%. This massive variance completely spans the IQRs of F, C, B, and A, proving that high attendance alone does not protect against low grade outcomes.


* **Scatter Plot & Regression Metric Evaluation:**
  * *Weak Linear explanatory Power:* A trend line $R^2$ of **0.06** proves that `Attendance Percentage` explains only ~6% of the continuous variance in `Final Exam Score`.
  * *Marginal Artifacts:* Bivariate marginal spikes at the upper boundaries (near 100% attendance and 100 final score) reflect individual feature caps rather than a strict linear trajectory across the full population.



**Analytical Takeaway:** Attendance functions as a weak linear predictor overall ($R^2 = 0.06$). While severe absenteeism guarantees lower scores and near-perfect attendance supports Grade A outcomes, attendance carries low predictive resolution for differentiating intermediate B, C, and D grades. Tree models will need to pair attendance with high-impact features like study hours to resolve these middle-tier ambiguities.

In [38]:
feature = num_features[3]

fig = px.histogram(
    df,
    x=feature,
    text_auto=True,
    title=" ".join(feature.split('_')).title() + " Histogram Distribution",
)
fig.show()
fig.data

(Histogram({
     'alignmentgroup': 'True',
     'bingroup': 'x',
     'hovertemplate': 'sleep_hours=%{x}<br>count=%{y}<extra></extra>',
     'legendgroup': '',
     'marker': {'color': '#636efa', 'pattern': {'shape': ''}},
     'name': '',
     'offsetgroup': '',
     'orientation': 'v',
     'showlegend': False,
     'texttemplate': '%{value}',
     'x': array([6.5, 5.7, 7.9, ..., 8. , 8.1, 6.1]),
     'xaxis': 'x',
     'yaxis': 'y'
 }),)

An analysis of the `Sleep Hours` histogram reveals a well-behaved, slightly bimodal distribution centered around healthy sleep habits, flanked by notable extreme tails:

* **Central Tendency & Bimodal Peaks:** The distribution concentrates heavily between 6.0 and 7.5 hours, exhibiting two distinct peaks in the sweet spot of healthy rest:
  * *First Peak:* 69 students at 6.4–6.5 hours of sleep.
  * *Second Peak:* 71 students at 7.2–7.3 hours of sleep.
  * *Domain Alignment:* This bimodal structure aligns well with recommended sleep guidelines, confirming that the vast majority of the student body maintains optimal rest cycles.


* **Extreme Lower Tail (Deprived Sleep):** A small cluster of severely sleep-deprived students spans the 3.0 to 4.2-hour range. Over 10 students report fewer than 4.2 hours of sleep, with sparse bin counts (e.g., 1 to 3 students per bin) down to the 3.0-hour minimum. Evaluating whether this acute sleep deficit degrades performance or reflects late-night cramming will be key.
* **Extreme Upper Tail (Excessive Sleep):** Conversely, a small tail extends up to 10.1 hours, including 5 students sleeping between 10.0 and 10.1 hours. Cross-referencing these high-sleep outliers against study hours will reveal if excessive rest correlates with academic disengagement.
* **Modeling Takeaway:** The overall spread (3.0 to 10.1 hours) provides a continuous, near-normal range without severe skewness. Like `Study Hours`, this feature requires minimal transformation and can be ingested directly into linear or tree-based regression models.

In [39]:
fig = px.box(
    df,
    x=feature,
    color=clf_target,
    title=" ".join(feature.split('_')).title() + " vs "  + " ".join(clf_target.split('_')).title() + " (Box Plot)",
)
fig.show()
print(fig.data)

fig = px.scatter(
    df,
    x=feature,
    y=reg_target,
    trendline='ols',
    marginal_x='histogram',
    marginal_y='histogram',
)
fig.show()

(Box({
    'alignmentgroup': 'True',
    'hovertemplate': 'final_grade=A<br>sleep_hours=%{x}<extra></extra>',
    'legendgroup': 'A',
    'marker': {'color': '#636efa'},
    'name': 'A',
    'notched': False,
    'offsetgroup': 'A',
    'orientation': 'h',
    'showlegend': True,
    'x': array([6.5, 5.7, 7.9, ..., 5.6, 5.9, 7.1]),
    'x0': ' ',
    'xaxis': 'x',
    'y0': ' ',
    'yaxis': 'y'
}), Box({
    'alignmentgroup': 'True',
    'hovertemplate': 'final_grade=B<br>sleep_hours=%{x}<extra></extra>',
    'legendgroup': 'B',
    'marker': {'color': '#EF553B'},
    'name': 'B',
    'notched': False,
    'offsetgroup': 'B',
    'orientation': 'h',
    'showlegend': True,
    'x': array([8. , 5.7, 6. , ..., 6.5, 5.6, 8.1]),
    'x0': ' ',
    'xaxis': 'x',
    'y0': ' ',
    'yaxis': 'y'
}), Box({
    'alignmentgroup': 'True',
    'hovertemplate': 'final_grade=D<br>sleep_hours=%{x}<extra></extra>',
    'legendgroup': 'D',
    'marker': {'color': '#00cc96'},
    'name': 'D',
    'notc

Evaluating `Sleep Hours` against final academic performance confirms that sleep duration holds virtually zero predictive value in this dataset, yielding disappointing but clear empirical results:

* **Box Plot Overlap:** Across every categorical letter grade (A through F), the interquartile ranges (IQRs) and median sleep hours overlap almost completely. No single grade tier demonstrates a statistically distinct sleep distribution, proving that rest duration alone fails to separate high performers from low performers.
* **Scatter Plot & Variance ($R^2$):** Bivariate analysis reveals a dispersed cloud with no discernible linear or non-linear trajectory. A trend line $R^2$ of **0.02** confirms that sleep duration explains a negligible 2% of the variance in `Final Exam Score`. Marginal frequency distributions further highlight uniform point scattering rather than functional clustering.
* **Domain Realities vs. Data Artifacts:** While domain intuition suggests that acute sleep deprivation or healthy rest cycles should impact cognitive performance, the data exhibits a synthetic detachment between sleep hours and exam outcomes.

**Analytical Takeaway:** `Sleep Hours` acts as noise within this predictive pipeline. Given its near-zero correlation ($R^2 = 0.02$) and heavy category overlap, tree-based models and linear regressors alike will gain minimal predictive signal from this feature.

In [40]:
feature = num_features[4]

fig = px.histogram(
    df,
    x=feature,
    text_auto=True,
    title=" ".join(feature.split('_')).title() + " Histogram Distribution",
)
fig.show()
fig.data

(Histogram({
     'alignmentgroup': 'True',
     'bingroup': 'x',
     'hovertemplate': 'previous_grade=%{x}<br>count=%{y}<extra></extra>',
     'legendgroup': '',
     'marker': {'color': '#636efa', 'pattern': {'shape': ''}},
     'name': '',
     'offsetgroup': '',
     'orientation': 'v',
     'showlegend': False,
     'texttemplate': '%{value}',
     'x': array([76.9, 75.5, 88.5, ..., 65.2, 52.2, 45.4]),
     'xaxis': 'x',
     'yaxis': 'y'
 }),)

An analysis of the **Previous Grade Histogram Distribution** reveals a well-defined, bell-shaped continuous feature profile with distinct behavioral peaks and tail behaviors:

* **Central Tendency & Dominant Peak:** The distribution follows a near-Gaussian, bell-shaped curve centered squarely in the 60–75 range. The single primary mode occurs right at **70** (`previous_grade`), reaching the global maximum bin count of **77 students**. Secondary surrounding peaks appear at ~62 (count: 58), ~66 (count: 60), and ~74 (count: 64), showing high student density in the mid-range performance band.
* **Lower Tail Dispersion:** A long, sparse left tail extends down toward the minimum boundary near 0–40. Bin counts below 40 drop off sharply into single digits (e.g., counts ranging from 1 to 3 students per bin), representing a tiny, isolated cohort of historically low-performing students.
* **Upper Boundary Spike (Ceiling Effect):** Moving up the upper scale, counts taper off naturally past 80 (e.g., bin `77–78.9` has a count of 47, dropping down to 11 at `92–94` and 5 at `98–99`). However, a sudden, artificial uptick occurs in the final bin at **100**, leaping back up to a count of **11 students**. This indicates a ceiling effect where historical grades were capped or rounded up at 100%.
* **Modeling Takeaway:** The overall symmetry and lack of severe skewness mean that `previous_grade` does not require logarithmic or power transformations. Its smooth continuous distribution—combined with strong domain relevance—makes it a primary candidate to serve as a high-signal linear or non-linear feature for predicting `Final Exam Score`.

In [41]:
fig = px.box(
    df,
    x=feature,
    color=clf_target,
    title=" ".join(feature.split('_')).title() + " vs "  + " ".join(clf_target.split('_')).title() + " (Box Plot)",
)
fig.show()
print(fig.data)

fig = px.scatter(
    df,
    x=feature,
    y=reg_target,
    trendline='ols',
    marginal_x='histogram',
    marginal_y='histogram',
)
fig.show()
fig.data

(Box({
    'alignmentgroup': 'True',
    'hovertemplate': 'final_grade=A<br>previous_grade=%{x}<extra></extra>',
    'legendgroup': 'A',
    'marker': {'color': '#636efa'},
    'name': 'A',
    'notched': False,
    'offsetgroup': 'A',
    'orientation': 'h',
    'showlegend': True,
    'x': array([76.9, 75.5, 88.5, ..., 78. , 75.4, 82.2]),
    'x0': ' ',
    'xaxis': 'x',
    'y0': ' ',
    'yaxis': 'y'
}), Box({
    'alignmentgroup': 'True',
    'hovertemplate': 'final_grade=B<br>previous_grade=%{x}<extra></extra>',
    'legendgroup': 'B',
    'marker': {'color': '#EF553B'},
    'name': 'B',
    'notched': False,
    'offsetgroup': 'B',
    'orientation': 'h',
    'showlegend': True,
    'x': array([85.1, 79.8, 78.4, ..., 81.8, 51. , 52.2]),
    'x0': ' ',
    'xaxis': 'x',
    'y0': ' ',
    'yaxis': 'y'
}), Box({
    'alignmentgroup': 'True',
    'hovertemplate': 'final_grade=D<br>previous_grade=%{x}<extra></extra>',
    'legendgroup': 'D',
    'marker': {'color': '#00cc96'},
    '

(Scatter({
     'hovertemplate': 'previous_grade=%{x}<br>final_exam_score=%{y}<extra></extra>',
     'legendgroup': '',
     'marker': {'color': '#636efa', 'symbol': 'circle'},
     'mode': 'markers',
     'name': '',
     'orientation': 'v',
     'showlegend': False,
     'x': array([76.9, 75.5, 88.5, ..., 65.2, 52.2, 45.4]),
     'xaxis': 'x',
     'y': array([100. , 100. ,  97.3, ...,  79.2,  82.2,  70.4]),
     'yaxis': 'y'
 }),
 Histogram({
     'alignmentgroup': 'True',
     'bingroup': 'x',
     'hovertemplate': 'previous_grade=%{x}<br>count=%{y}<extra></extra>',
     'legendgroup': '',
     'marker': {'color': '#636efa'},
     'name': '',
     'offsetgroup': '',
     'opacity': 0.5,
     'showlegend': False,
     'x': array([76.9, 75.5, 88.5, ..., 65.2, 52.2, 45.4]),
     'xaxis': 'x3',
     'yaxis': 'y3'
 }),
 Histogram({
     'alignmentgroup': 'True',
     'bingroup': 'y',
     'hovertemplate': 'final_exam_score=%{y}<br>count=%{x}<extra></extra>',
     'legendgroup': '',
    

Analyzing `Previous Grade` against `Final Exam Score` across both bivariate continuous scatter plots and categorical box plots confirms a strong, positive baseline correlation alongside key structural artifacts:

* **Bivariate Continuous Dynamics (Scatter Plot Insights):**
* *Linear Slope & Correlation:* The regression trend line exhibits a steady upward trajectory, demonstrating that students with higher historical baseline grades (`previous_grade`) reliably yield higher current exam outcomes (`final_exam_score`). With $R^2 = 0.16$, it acts as our second-strongest linear feature overall.
* *Boundary Capping & Dispersion:* Bivariate marginal histograms reveal pronounced ceiling effects on both axes—a top spike at `previous_grade = 100` and a dense horizontal pooling at `final_exam_score = 100`. The wide vertical dispersion along the middle-performance band (50–80 `previous_grade`) shows that past grades alone do not dictate final scores.


* **Categorical Grade Stratification (Box Plot Insights):**
* *Extreme Grade Separation:* The box plots reflect a clear upward progression at the macro level. **Grade F** students occupy the lowest median baseline (~60.5), while **Grade A** students hold the highest (~76.0 with an IQR of ~67.0–85.5).
* *Middle-Tier Inversion (Grade C vs. Grade D):* An interesting non-monotonic overlap occurs between intermediate tiers. Students earning **Grade D** show a slightly *higher* median past grade (~62.8) than those earning **Grade C** (~64.5 median, but with a lower $Q_1$ bound extending to 57.5).
* *High-Volatility Outliers:* Severe low-end outliers exist in **Grade B** (past grades down to ~33.5 and 37.5) and **Grade C** (outliers at ~31.0, 94.5, and 99.0), proving that some students experience extreme performance swings regardless of historical baseline.



**Analytical Takeaway:** `Previous Grade` serves as a vital linear anchor for student performance ($R^2 = 0.16$). However, because middle-tier grade inversions and ceiling boundary caps disrupt pure linearity, pairing `previous_grade` with active behavioral features (`study_hours`, `attendance`) inside tree-based models will be essential to map these non-linear interactions accurately.

In [42]:
corr = df.select_dtypes(exclude='object').corr(method='spearman')
fig = px.imshow(corr, text_auto=True, height=800, title="Numerical features Spearman Correlation Heatmap")
fig.show()
fig.data

(Heatmap({
     'coloraxis': 'coloraxis',
     'hovertemplate': 'x: %{x}<br>y: %{y}<br>color: %{z}<extra></extra>',
     'name': '0',
     'texttemplate': '%{z}',
     'x': array(['student_id', 'study_time_hours', 'attendance_percent', 'sleep_hours',
                 'previous_grade', 'final_exam_score'], dtype=object),
     'xaxis': 'x',
     'y': array(['student_id', 'study_time_hours', 'attendance_percent', 'sleep_hours',
                 'previous_grade', 'final_exam_score'], dtype=object),
     'yaxis': 'y',
     'z': array([[ 1.        ,  0.02958332, -0.04274459, -0.00812547,  0.01169514,
                   0.02416464],
                 [ 0.02958332,  1.        , -0.05968006,  0.01633325, -0.03103093,
                   0.56128809],
                 [-0.04274459, -0.05968006,  1.        ,  0.03580259, -0.01074864,
                   0.25167609],
                 [-0.00812547,  0.01633325,  0.03580259,  1.        ,  0.00288055,
                   0.14559128],
                 [ 0.

Evaluating the Spearman rank correlation matrix for our numerical features against `Final Exam Score` yields crucial insights into non-linear dependencies and feature independence:

* **Non-Linear Relationships with Target (`Final Exam Score`):**
  * *`Study Hours` ($r_s = 0.56$):* Demonstrates the strongest monotonic relationship with final performance. Capturing non-linear rankings lifts its signal significantly compared to its linear fit ($R^2 = 0.32$).
  * *`Previous Grade` ($r_s = 0.39$):* Reaffirms its role as a solid secondary anchor, capturing baseline academic capability across ordinal grade ranks ($R^2 = 0.16$).
  * *`Attendance Percentage` ($r_s = 0.25$):* Shows a moderate monotonic rank correlation, proving far more useful here than in linear evaluation ($R^2 = 0.06$) by capturing extreme-tail risks.



* **Lack of Multicollinearity:** Inter-feature correlations among predictors are near zero across the board. This indicates that our input variables are virtually mutually exclusive—study habits, past academic records, and attendance operate as independent signals rather than redundant inputs.
* **Data Cleaning Note:** `Student ID` exhibits arbitrary correlation values due to its numeric index nature. As an artificial identifier with zero domain signal, it will be dropped prior to pipeline training.

**Analytical Takeaway:** The absence of multicollinearity simplifies feature selection, while strong non-linear Spearman coefficients ($0.56$, $0.39$, $0.25$) confirm that tree-based algorithms will successfully extract complementary, non-redundant signals from these independent predictors.

In [43]:
# Removing Student ID permanently
if 'student_id' in df.columns:
    df = df.drop(columns=['student_id'])

In [44]:
# Study Time Hours Extreme
highest_study_time_std = df[df.study_time_hours == 8.1]
lowest_study_time_std = df[df.study_time_hours == 0.5]

highest_study_time_std = highest_study_time_std.drop(columns=['study_time_hours'])
lowest_study_time_std = lowest_study_time_std.drop(columns=['study_time_hours'])

# Number of extremes
print('Number of Higher Extremes: ', highest_study_time_std.shape[0])
print('Number of Lower Extremes : ', lowest_study_time_std.shape[0])

Number of Higher Extremes:  1
Number of Lower Extremes :  23


An analysis of extreme boundary behavior in `Study Hours` highlights a key structural asymmetry in sample sizes:

* **High-Effort Extremes ($N = 1$):** A single isolated student sits at the top end, studying ~8 hours daily. As an isolated maximum outlier, this data point reflects rare, peak effort. It can easily be evaluated through individual case analysis without distorting global model metrics.
* **Low-Effort Extremes ($N = 23$):** Conversely, 23 students concentrate at the ultra-low boundary ($\le 30$ minutes daily). Rather than acting as random point anomalies, this sizeable cluster forms a distinct, low-engagement sub-cohort within the student population.
* **Analytical Treatment:** Because 23 students represent ~2.3% of the overall sample, treating them as simple data errors or trimming them as noise would destroy valid structural information.

**Analytical Takeaway:** Extreme low-effort study behavior is a robust, population-level phenomenon rather than a measurement artifact. The predictive pipeline must retain these 23 points, leveraging tree-based splits to model whether this cohort represents severe academic risk or exceptional learning efficiency.

In [45]:
high_study = highest_study_time_std.select_dtypes(exclude='object').mean()
low_study_mean = lowest_study_time_std.select_dtypes(exclude='object').mean()

comparison_df = pd.DataFrame({
    'High Study Time (8.1 Hrs)': high_study,
    'Low Study Time Avg (0.5 Hrs)': low_study_mean
}).reset_index().rename(columns={'index': 'Feature'})

df_melted = comparison_df.melt(
    id_vars='Feature',
    value_vars=['High Study Time (8.1 Hrs)', 'Low Study Time Avg (0.5 Hrs)'],
    var_name='Cohort',
    value_name='Value'
)

fig = px.bar(
    df_melted,
    x='Feature',
    y='Value',
    color='Cohort',
    barmode='group',
    title="Feature Profile Comparison: 8.1-Hour High Effort vs. 0.5-Hour Low Effort Cohort",
    text_auto='.2f'
)

fig.update_layout(
    xaxis_title="Numerical Features",
    yaxis_title="Value / Standardized Metric",
    legend_title="Student Group",
    template="plotly_white"
)

fig.show()

A comparative analysis of our extreme study cohorts uncovers striking dynamics surrounding time investment, academic leverage, and diminishing returns:

* **High-Effort Outlier Profile (8.1 Hrs Daily Study):**
  * *Metrics:* `Study Hours` = 8.1 | `Attendance` = 76.0% | `Sleep Hours` = 5.0 | `Previous Grade` = 74.4 | `Final Exam Score` = 98.3
  * *Behavioral Signature:* This student sacrifices rest (5 hours of sleep) and maintains average attendance (76%) to brute-force an elite score (98.3) through sheer, dedicated volume of study hours.


* **Low-Effort Cohort Baseline (0.5 Hrs Daily Study Avg, N=23):**
  * *Metrics:* `Study Hours` = 0.5 | `Attendance` = 85.0% | `Sleep Hours` = 7.0 | `Previous Grade` = 71.0 | `Final Exam Score` = 71.2
  * *Behavioral Signature:* These 23 students maintain healthy lifestyle habits (7 hours of sleep) and high classroom engagement (85% attendance), resting on a historical baseline (~71.0 previous grade) nearly identical to the 8.1-hour outlier.


* **Efficiency & Return on Investment (ROI):**
  * *Diminishing Marginal Returns:* Increasing daily study effort from 0.5 to 8.1 hours represents a **1,520% increase in study time**, yet it produces only a **38% increase in final exam score** (71.2 to 98.3).
  * *Baseline Floor:* High attendance (85%) combined with past domain knowledge (~71.0 previous grade) creates a strong performance floor (~71.2 final score). This allows low-effort students to remain fully viable (Grade C/B tier) with virtually zero outside study time.



**Analytical Takeaway:** Prior baseline knowledge and class attendance set a resilient performance floor, ensuring that zero-study students do not automatically fail. Meanwhile, pushing into top-tier scores (A/A+ status) requires massive dedicated effort, but incurs heavy diminishing returns and severe sleep tradeoffs.

In [46]:
highest_study_time_std

,gender,attendance_percent,sleep_hours,parental_education,internet_access,extracurricular_activities,part_time_job,previous_grade,final_exam_score,final_grade
92,Female,76.1,5.2,Bachelors,Yes,Yes,No,74.4,98.3,A


Expanding the profile of our 8.1-hour high-effort outlier to include categorical attributes provides a complete, multi-dimensional view of top-tier performance:

* **Categorical Profile (8.1 Hrs Study Student):**
  * *Socioeconomic & Support Factors:* **Parental Education** = Bachelor’s Degree | **Internet Access** = Yes | **Part-Time Job** = No
  * *Engagement Factors:* **Gender** = Female | **Extracurricular Activities** = Yes
  * *Quantitative Performance:* **Study Hours** = 8.1 | **Previous Grade** = 74.4 | **Final Exam Score** = 98.3


* **Environment & Household Leverage:**
An educated household baseline (Bachelor's) combined with home internet access provides strong academic infrastructure. Crucially, the absence of a part-time job frees up the time budget required to sustain an 8.1-hour daily study schedule without financial distraction.
* **Holistic Balance:**
Active participation in extracurricular activities demonstrates that her high-effort strategy is not isolated academic burnout, but rather a structured routine that pairs intense home study with active school involvement to secure an elite score (98.3).

**Analytical Takeaway:** Elite academic performance ($98.3$) is enabled by a supportive environment: high parental education, reliable internet, and zero employment drag provide the temporal and structural runway needed to translate massive study hours into peak results.

In [47]:
for cat in cat_features:
    formatted_title = cat.replace('_', ' ').title()
    
    fig = px.pie(
        lowest_study_time_std, 
        names=cat, 
        hole=0.4, 
        title=f"Distribution of {formatted_title} (Low-Effort Cohort: 0.5 Hrs Study)"
    )
    fig.update_traces(textinfo='label+percent')
    fig.show()

Analyzing the categorical profile of the low-effort cohort ($N=23$, $\le 0.5$ hours/day study time) reveals striking demographic balances, key environmental factors, and final performance distributions:

* **Gender Balance (Countering Regional Bias):**
  * *Distribution:* **52.2% Male** | **47.8% Female**
  * *Insight:* Contrary to common regional stereotypes that assume low study effort is heavily skewed toward male students, the cohort displays an almost even 50/50 split. Gender provides zero explanatory power regarding low study engagement in this dataset.


* **Parental Education Profile:**
  * *Distribution:* **High School (34.8%)** | **Bachelor’s (34.8%)** | **Master’s (13.0%)** | **Not Disclosed (13.0%)** | **Ph.D. (4.3%)**
  * *Insight:* Low effort is not confined to low-income or uneducated households. While 34.8% have High School-educated parents, an identical 34.8% come from Bachelor's households, and over 17% come from advanced postgraduate backgrounds (Master's/Ph.D.).


* **Access & Time-Use Dynamics:**
  * *Internet Access:* **82.6% Yes** | **17.4% No** — High connectivity confirms that lack of study time is driven by personal allocation, not an absence of learning resources.
  * *Extracurriculars vs. Employment:* **73.9% participate in Extracurriculars**, whereas **73.9% do NOT hold a Part-Time Job** (only 26.1% work). This inverse structural match proves that low study time stems from active participation in school activities and personal leisure rather than work-related time poverty.


* **Final Grade Outcomes (Efficiency Tradeoff):**
  * *Grade Distribution:* **Grade D (43.5%)** leads, followed by **Grade C (30.4%)**, **Grade B (17.4%)**, and **Grade F (8.7%)**.
  * *Performance Reality:* While a **Grade D** is objectively poor, earning a passing grade (C/D tier) with only 30 minutes of daily study highlights remarkable baseline efficiency, enabled heavily by their high attendance (85%) and past knowledge foundation.



**Analytical Takeaway:** Synthetic dataset indicators emerge in the rigid categorical proportions (e.g., matching 73.9% splits across jobs and activities). Nevertheless, the profile shows that low-effort students are well-connected, highly involved in school activities, and leverage high attendance to barely secure passing grades without home study.

In [48]:
# Extreme students individual analysis
df.describe()

,study_time_hours,attendance_percent,sleep_hours,previous_grade,final_exam_score
count,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000
mean,3.570700,85.092300,6.799500,69.740900,83.543500
std,1.478559,9.270685,1.203527,12.613425,10.341333
min,0.500000,54.800000,3.200000,31.300000,46.800000
25%,2.600000,78.800000,5.900000,61.000000,76.075000
50%,3.600000,85.200000,6.800000,69.600000,83.800000
75%,4.500000,91.900000,7.600000,78.400000,91.525000
max,8.100000,100.000000,10.000000,100.000000,100.000000


In [49]:
# Attandance extremes
high_attendance_stds = df[df.attendance_percent == 100]
low_attendance_stds = df[df.attendance_percent == 54.8]

# Number of extremes
print('Number of Higher Extremes: ', high_attendance_stds.shape[0])
print('Number of Lower Extremes : ', low_attendance_stds.shape[0])

Number of Higher Extremes:  70
Number of Lower Extremes :  1


## **Data Preprocessing**

In [50]:
# 1. Separate Features (X) and Target (y)
y = df['final_exam_score']
y_clf = df['final_grade']
X = df.drop(columns=['final_grade', 'final_exam_score'])

# 2. Feature Group Definitions
binary_columns = ['gender', 'internet_access', 'extracurricular_activities', 'part_time_job']
ordinal_columns = ['parental_education']
numerical_columns = ['study_time_hours', 'previous_grade', 'attendance_percent', 'sleep_hours']

# 3. Explicit hierarchy matching exact dataset strings
# Includes 'Not Disclosed' and uses 'Bachelors'/'Masters' without apostrophes
parental_ed_order = [
    'Not Disclosed', 
    'High School', 
    'Associate', 
    'Bachelors', 
    'Masters', 
    'PhD'
]

# 4. Build ColumnTransformer Pipeline
preprocessor = ColumnTransformer(
    transformers=[
        # Binary categorical encoding
        ('binary_enc', OrdinalEncoder(), binary_columns),
            
        # Ordinal categorical encoding with explicit hierarchy
        ('ordinal_enc', OrdinalEncoder(categories=[parental_ed_order]), ordinal_columns),
        
        # Outlier-robust scaling for numerical features
        ('num_scaler', RobustScaler(), numerical_columns)
    ],
    remainder='drop'
)

# 5. Build ColumnTransformer Pipeline for Tree Models
tree_preprocessor = ColumnTransformer(
    transformers=[
        # Binary categorical encoding
        ('binary_enc', OrdinalEncoder(), binary_columns),
            
        # Ordinal categorical encoding with explicit hierarchy
        ('ordinal_enc', OrdinalEncoder(categories=[parental_ed_order]), ordinal_columns),  
    ],
    remainder='passthrough'
)

In [51]:
# Hardcoded Grade Boundary Function
def convert_scores_to_grades(scores):
    """Maps continuous numerical exam scores to letter grades."""
    bins = [-float('inf'), 60, 70, 80, 90, float('inf')]
    labels = ['F', 'D', 'C', 'B', 'A']
    return pd.cut(scores, bins=bins, labels=labels, right=False)

# **Models**

In [52]:
# Traditional ML models requiring preprocessing (scaling & categorical encoding)
traditional_models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(alpha=1.0),
    "Lasso Regression": Lasso(alpha=0.1),
    "ElasticNet": ElasticNet(alpha=0.1, l1_ratio=0.5),
    "Huber Regressor": HuberRegressor(),
    "Bayesian Ridge": BayesianRidge(),
    "Support Vector Regressor (SVR)": SVR(kernel='rbf', C=1.0),
    "K-Nearest Neighbors (KNN)": KNeighborsRegressor(n_neighbors=5)
}

# Ground truth letter grades derived directly from true continuous y
y_true_grades = convert_scores_to_grades(y)

fitted_production_pipelines = {}
traditional_results = []
kf = KFold(n_splits=5, shuffle=True, random_state=42)

for name, model in traditional_models.items():
    fold_train_rmse, fold_val_rmse, fold_val_mae, fold_val_r2 = [], [], [], []
    fold_val_acc, fold_val_precision, fold_val_recall, fold_val_f1 = [], [], [], []
    
    for train_idx, val_idx in kf.split(X, y):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
        
        # FIX: Instantiate a completely fresh pipeline per fold 
        # so RobustScaler computes statistics ONLY on X_train
        fold_pipeline = Pipeline(steps=[
            ('preprocessor', preprocessor), # Safe from data leakage now
            ('regressor', model)
        ])
        
        fold_pipeline.fit(X_train, y_train)
        
        # Train RMSE (Overfitting check)
        y_train_pred = fold_pipeline.predict(X_train)
        fold_train_rmse.append(root_mean_squared_error(y_train, y_train_pred))
        
        # Out-of-Fold Validation
        y_val_pred = fold_pipeline.predict(X_val)
        
        fold_val_rmse.append(root_mean_squared_error(y_val, y_val_pred))
        fold_val_mae.append(mean_absolute_error(y_val, y_val_pred))
        fold_val_r2.append(r2_score(y_val, y_val_pred))
        
        # Classification Metric Tracking
        y_val_true_grades = convert_scores_to_grades(y_val)
        y_val_pred_grades = convert_scores_to_grades(y_val_pred)
        
        acc = accuracy_score(y_val_true_grades, y_val_pred_grades)
        precision, recall, f1, _ = precision_recall_fscore_support(
            y_val_true_grades, y_val_pred_grades, average='weighted', zero_division=0
        )
        
        fold_val_acc.append(acc)
        fold_val_precision.append(precision)
        fold_val_recall.append(recall)
        fold_val_f1.append(f1)
        
    # Final Production Pipeline fit on 100% data
    production_pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('regressor', model)
    ])
    production_pipeline.fit(X, y)
    fitted_production_pipelines[name] = production_pipeline
    
    traditional_results.append({
        'Model': name,
        'Train RMSE': np.mean(fold_train_rmse),
        'Test RMSE (OOF)': np.mean(fold_val_rmse),
        'RMSE Gap (Test - Train)': np.mean(fold_val_rmse) - np.mean(fold_train_rmse),
        'Test MAE (OOF)': np.mean(fold_val_mae),
        'Test R² (OOF)': np.mean(fold_val_r2),
        'Grade Accuracy (OOF)': np.mean(fold_val_acc),
        'Grade Precision (OOF)': np.mean(fold_val_precision),
        'Grade Recall (OOF)': np.mean(fold_val_recall),
        'Grade F1-Score (OOF)': np.mean(fold_val_f1)
    })

traditional_results_df = (
    pd.DataFrame(traditional_results)
    .sort_values(by='Test RMSE (OOF)')
    .reset_index(drop=True)
)

traditional_results_df

,Model,Train RMSE,Test RMSE (OOF),RMSE Gap (Test - Train),Test MAE (OOF),Test R² (OOF),Grade Accuracy (OOF),Grade Precision (OOF),Grade Recall (OOF),Grade F1-Score (OOF)
0,Bayesian Ridge,5.974449,6.052757,0.078308,4.801785,0.652594,0.588,0.605351,0.588,0.580984
1,Ridge Regression,5.974255,6.052758,0.078503,4.801165,0.652592,0.586,0.602311,0.586,0.578909
2,Linear Regression,5.974183,6.053012,0.078829,4.800741,0.652560,0.586,0.602311,0.586,0.578909
3,Huber Regressor,5.977703,6.055702,0.077998,4.810616,0.652246,0.581,0.597244,0.581,0.574717
4,Lasso Regression,5.996547,6.072056,0.075509,4.824625,0.650341,0.581,0.603509,0.581,0.573057
5,ElasticNet,6.083657,6.151911,0.068254,4.893903,0.641228,0.584,0.615918,0.584,0.570280
6,Support Vector Regressor (SVR),6.270374,6.459898,0.189524,5.167398,0.604668,0.558,0.579963,0.558,0.536719
7,K-Nearest Neighbors (KNN),6.047576,7.657829,1.610253,6.115640,0.445063,0.506,0.512292,0.506,0.488022


Looking at overall model performance, a few algorithms show promise, but we need to acknowledge a core limitation: our features are relatively weak predictors of the target. Whether we predict continuous exam scores or discrete letter grades—two sides of the same coin—our earlier scatter plots and correlation matrices revealed that variables like study hours or attendance lack strong linear relationships with final outcomes. While these factors intuitively matter, their empirical signal here is modest.

Furthermore, with such a small dataset, drawing definitive conclusions is tricky. Every sample counts, which is why we relied strictly on 5-Fold Cross-Validation rather than a standard train-test split—losing even $20\%$ of our data to a static holdout set strips away vital signal.What we are seeing now—an average test $R^2$ capping around $0.65$ and out-of-fold grade accuracy peaking near $58\text{--}59\%$—likely represents the ceiling for these baseline architectures. Even with aggressive hyperparameter optimization, the fundamental noise floor and limited sample size will strictly constrain further gains.

In [53]:
# 1. Format fitted production pipelines into (name, estimator) tuples
estimators_list = [(name, pipeline) for name, pipeline in fitted_production_pipelines.items()]

# 2. Setup KFold and tracking lists
kf = KFold(n_splits=5, shuffle=True, random_state=42)

fold_train_rmse, fold_val_rmse, fold_val_mae, fold_val_r2 = [], [], [], []
fold_val_acc, fold_val_precision, fold_val_recall, fold_val_f1 = [], [], [], []

# 3. Out-Of-Fold Cross-Validation Loop
for train_idx, val_idx in kf.split(X, y):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    # Instantiate fresh clones of underlying base estimators for this fold
    fold_estimators = [(name, clone(est)) for name, est in estimators_list]
    fold_voting_reg = VotingRegressor(estimators=fold_estimators)
    
    # Fit voting regressor ONLY on training fold
    fold_voting_reg.fit(X_train, y_train)
    
    # Train RMSE (Overfitting diagnostic)
    y_train_pred = fold_voting_reg.predict(X_train)
    fold_train_rmse.append(root_mean_squared_error(y_train, y_train_pred))
    
    # Out-of-Fold Validation Predictions
    y_val_pred = fold_voting_reg.predict(X_val)
    
    # Out-of-Fold Regression Metrics
    fold_val_rmse.append(root_mean_squared_error(y_val, y_val_pred))
    fold_val_mae.append(mean_absolute_error(y_val, y_val_pred))
    fold_val_r2.append(r2_score(y_val, y_val_pred))
    
    # Out-of-Fold Converted Grade Classification Metrics
    y_val_true_grades = convert_scores_to_grades(y_val)
    y_val_pred_grades = convert_scores_to_grades(y_val_pred)
    
    acc = accuracy_score(y_val_true_grades, y_val_pred_grades)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_val_true_grades, 
        y_val_pred_grades, 
        average='weighted', 
        zero_division=0
    )
    
    fold_val_acc.append(acc)
    fold_val_precision.append(precision)
    fold_val_recall.append(recall)
    fold_val_f1.append(f1)

# 4. Extract Mean Out-Of-Fold Metrics
voting_metrics = {
    'Model': 'Voting Regressor (Ensemble)',
    # --- Out-of-Fold Regression Metrics ---
    'Train RMSE': np.mean(fold_train_rmse),
    'Test RMSE (OOF)': np.mean(fold_val_rmse),
    'RMSE Gap (Test - Train)': np.mean(fold_val_rmse) - np.mean(fold_train_rmse),
    'Test MAE (OOF)': np.mean(fold_val_mae),
    'Test R² (OOF)': np.mean(fold_val_r2),
    # --- Out-of-Fold Classification Metrics ---
    'Grade Accuracy (OOF)': np.mean(fold_val_acc),
    'Grade Precision (OOF)': np.mean(fold_val_precision),
    'Grade Recall (OOF)': np.mean(fold_val_recall),
    'Grade F1-Score (OOF)': np.mean(fold_val_f1)
}

# 5. Fit Final Production Voting Regressor on 100% of data and store
production_estimators = [(name, clone(est)) for name, est in estimators_list]
production_voting_reg = VotingRegressor(estimators=production_estimators)
production_voting_reg.fit(X, y)
fitted_production_pipelines['Voting Regressor'] = production_voting_reg

# 6. Append to master dataframe and sort by Test RMSE (OOF)
traditional_results_df = (
    pd.concat([traditional_results_df, pd.DataFrame([voting_metrics])], ignore_index=True)
    .sort_values(by='Test RMSE (OOF)')
    .reset_index(drop=True)
)

# Display final comparison table
traditional_results_df

,Model,Train RMSE,Test RMSE (OOF),RMSE Gap (Test - Train),Test MAE (OOF),Test R² (OOF),Grade Accuracy (OOF),Grade Precision (OOF),Grade Recall (OOF),Grade F1-Score (OOF)
0,Bayesian Ridge,5.974449,6.052757,0.078308,4.801785,0.652594,0.588,0.605351,0.588,0.580984
1,Ridge Regression,5.974255,6.052758,0.078503,4.801165,0.652592,0.586,0.602311,0.586,0.578909
2,Linear Regression,5.974183,6.053012,0.078829,4.800741,0.652560,0.586,0.602311,0.586,0.578909
3,Huber Regressor,5.977703,6.055702,0.077998,4.810616,0.652246,0.581,0.597244,0.581,0.574717
4,Lasso Regression,5.996547,6.072056,0.075509,4.824625,0.650341,0.581,0.603509,0.581,0.573057
5,Voting Regressor (Ensemble),5.871545,6.136818,0.265273,4.864919,0.642970,0.588,0.607911,0.588,0.575371
6,ElasticNet,6.083657,6.151911,0.068254,4.893903,0.641228,0.584,0.615918,0.584,0.570280
7,Support Vector Regressor (SVR),6.270374,6.459898,0.189524,5.167398,0.604668,0.558,0.579963,0.558,0.536719
8,K-Nearest Neighbors (KNN),6.047576,7.657829,1.610253,6.115640,0.445063,0.506,0.512292,0.506,0.488022


Now that we have created a voting ensemble let's look at how the voting ensemble is performing as compared to the rest of the models consider that voting ensemble is basically an average or in this particular scenario so essentially we are taking an average of all the rest of the models predictions and we get an R square of 65 or 0.65 which is expected because the rest of the models have the same value so the average is not going to deviate much or it's not gonna you know shrink down much or grow much right if you look at the accuracy then this is the only model who has a conclusive 60% accuracy So clearly from an accuracy point of view this model is performing way better than the rest of the models We already know that the other models are able to perform quite well.

In [54]:
# 1. Format fitted production pipelines into (name, estimator) tuples
estimators_list = [(name, pipeline) for name, pipeline in fitted_production_pipelines.items()]

# 2. Setup KFold and tracking lists
kf = KFold(n_splits=5, shuffle=True, random_state=42)

fold_train_rmse, fold_val_rmse, fold_val_mae, fold_val_r2 = [], [], [], []
fold_val_acc, fold_val_precision, fold_val_recall, fold_val_f1 = [], [], [], []

# 3. Out-Of-Fold Cross-Validation Loop
for train_idx, val_idx in kf.split(X, y):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    # Instantiate a fresh VotingRegressor per fold
    fold_voting_reg = VotingRegressor(estimators=estimators_list)
    fold_voting_reg.fit(X_train, y_train)
    
    # Train RMSE (Overfitting diagnostic)
    y_train_pred = fold_voting_reg.predict(X_train)
    fold_train_rmse.append(root_mean_squared_error(y_train, y_train_pred))
    
    # Out-of-Fold Validation Predictions
    y_val_pred = fold_voting_reg.predict(X_val)
    
    # Regression Metrics (OOF)
    fold_val_rmse.append(root_mean_squared_error(y_val, y_val_pred))
    fold_val_mae.append(mean_absolute_error(y_val, y_val_pred))
    fold_val_r2.append(r2_score(y_val, y_val_pred))
    
    # Classification Metrics (OOF Converted Grades)
    y_val_true_grades = convert_scores_to_grades(y_val)
    y_val_pred_grades = convert_scores_to_grades(y_val_pred)
    
    acc = accuracy_score(y_val_true_grades, y_val_pred_grades)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_val_true_grades, 
        y_val_pred_grades, 
        average='weighted', 
        zero_division=0
    )
    
    fold_val_acc.append(acc)
    fold_val_precision.append(precision)
    fold_val_recall.append(recall)
    fold_val_f1.append(f1)

# 4. Extract Mean Out-Of-Fold Metrics
voting_metrics = {
    'Model': 'Voting Regressor (Ensemble)',
    # --- Out-of-Fold Regression Metrics ---
    'Train RMSE': np.mean(fold_train_rmse),
    'Test RMSE (OOF)': np.mean(fold_val_rmse),
    'RMSE Gap (Test - Train)': np.mean(fold_val_rmse) - np.mean(fold_train_rmse),
    'Test MAE (OOF)': np.mean(fold_val_mae),
    'Test R² (OOF)': np.mean(fold_val_r2),
    # --- Out-of-Fold Classification Metrics ---
    'Grade Accuracy (OOF)': np.mean(fold_val_acc),
    'Grade Precision (OOF)': np.mean(fold_val_precision),
    'Grade Recall (OOF)': np.mean(fold_val_recall),
    'Grade F1-Score (OOF)': np.mean(fold_val_f1)
}

# 5. Fit Final Production Voting Regressor on 100% of data and store
production_voting_reg = VotingRegressor(estimators=estimators_list)
production_voting_reg.fit(X, y)
fitted_production_pipelines['Voting Regressor'] = production_voting_reg

# 6. Append to master dataframe and sort by Test RMSE (OOF)
traditional_results_df = (
    pd.concat([traditional_results_df, pd.DataFrame([voting_metrics])], ignore_index=True)
    .sort_values(by='Test RMSE (OOF)')
    .reset_index(drop=True)
)

# Display final comparison table
traditional_results_df

,Model,Train RMSE,Test RMSE (OOF),RMSE Gap (Test - Train),Test MAE (OOF),Test R² (OOF),Grade Accuracy (OOF),Grade Precision (OOF),Grade Recall (OOF),Grade F1-Score (OOF)
0,Bayesian Ridge,5.974449,6.052757,0.078308,4.801785,0.652594,0.588,0.605351,0.588,0.580984
1,Ridge Regression,5.974255,6.052758,0.078503,4.801165,0.652592,0.586,0.602311,0.586,0.578909
2,Linear Regression,5.974183,6.053012,0.078829,4.800741,0.652560,0.586,0.602311,0.586,0.578909
3,Huber Regressor,5.977703,6.055702,0.077998,4.810616,0.652246,0.581,0.597244,0.581,0.574717
4,Lasso Regression,5.996547,6.072056,0.075509,4.824625,0.650341,0.581,0.603509,0.581,0.573057
5,Voting Regressor (Ensemble),5.871545,6.136818,0.265273,4.864919,0.642970,0.588,0.607911,0.588,0.575371
6,Voting Regressor (Ensemble),5.871545,6.136818,0.265273,4.864919,0.642970,0.588,0.607911,0.588,0.575371
7,ElasticNet,6.083657,6.151911,0.068254,4.893903,0.641228,0.584,0.615918,0.584,0.570280
8,Support Vector Regressor (SVR),6.270374,6.459898,0.189524,5.167398,0.604668,0.558,0.579963,0.558,0.536719
9,K-Nearest Neighbors (KNN),6.047576,7.657829,1.610253,6.115640,0.445063,0.506,0.512292,0.506,0.488022


Analyzing the VotingRegressor—which serves simultaneously as our continuous score predictor and discretized grade classifier—we observe solid overall performance. However, its metrics land on the conservative end of our evaluation spectrum. Because a soft voting ensemble operates as an averager, it aggregates outputs across all base estimators. While this effectively dampens variance, lower-confidence or underperforming predictions pull down the top-end results, causing a cancellation effect where extreme over-predictions and under-predictions neutralize each other.

While we could construct a targeted ensemble restricted to our top three or top five models, the fundamental performance ceiling remains unchanged. The data limitations establish a hard cap—peaking near $59\%$ out-of-fold grade accuracy—and ensembling alone cannot transcend the informational boundaries of our feature set.

In [55]:
# 1. Instantiate Regularized Decision Tree & Random Forest Models wrapped in Pipelines
# (tree_preprocessor handles encoding for non-boosting tree models)
tree_models = {
    "Decision Tree": Pipeline(steps=[
        ('preprocessor', tree_preprocessor),
        ('regressor', DecisionTreeRegressor(
            max_depth=4,
            min_samples_split=10,
            min_samples_leaf=5,
            random_state=42
        ))
    ]),
    "Random Forest": Pipeline(steps=[
        ('preprocessor', tree_preprocessor),
        ('regressor', RandomForestRegressor(
            n_estimators=100,
            max_depth=4,
            min_samples_split=10,
            min_samples_leaf=5,
            max_features=0.8,
            random_state=42
        ))
    ]),
    "Extra Trees": Pipeline(steps=[
        ('preprocessor', tree_preprocessor),
        ('regressor', ExtraTreesRegressor(
            n_estimators=100,
            max_depth=4,
            min_samples_split=10,
            min_samples_leaf=5,
            max_features=0.8,
            random_state=42
        ))
    ])
}

tree_results = []
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# 2. Out-of-Fold Validation Loop for Tree Models
for name, pipeline in tree_models.items():
    fold_train_rmse, fold_val_rmse, fold_val_mae, fold_val_r2 = [], [], [], []
    fold_val_acc, fold_val_precision, fold_val_recall, fold_val_f1 = [], [], [], []
    
    for train_idx, val_idx in kf.split(X, y):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
        
        # Clone pipeline instance for clean fold fitting (prevents preprocessor & state leak)
        fold_pipeline = clone(pipeline)
        fold_pipeline.fit(X_train, y_train)
        
        # Train RMSE (Overfitting diagnostic)
        y_train_pred = fold_pipeline.predict(X_train)
        fold_train_rmse.append(root_mean_squared_error(y_train, y_train_pred))
        
        # Out-of-Fold Validation Predictions
        y_val_pred = fold_pipeline.predict(X_val)
        
        # Out-of-Fold Regression Metrics
        fold_val_rmse.append(root_mean_squared_error(y_val, y_val_pred))
        fold_val_mae.append(mean_absolute_error(y_val, y_val_pred))
        fold_val_r2.append(r2_score(y_val, y_val_pred))
        
        # Out-of-Fold Converted Grade Classification Metrics
        y_val_true_grades = convert_scores_to_grades(y_val)
        y_val_pred_grades = convert_scores_to_grades(y_val_pred)
        
        acc = accuracy_score(y_val_true_grades, y_val_pred_grades)
        precision, recall, f1, _ = precision_recall_fscore_support(
            y_val_true_grades, 
            y_val_pred_grades, 
            average='weighted', 
            zero_division=0
        )
        
        fold_val_acc.append(acc)
        fold_val_precision.append(precision)
        fold_val_recall.append(recall)
        fold_val_f1.append(f1)
        
    # Fit pipeline on 100% of data for production storage
    production_pipeline = clone(pipeline)
    production_pipeline.fit(X, y)
    fitted_production_pipelines[name] = production_pipeline
    
    # Store aggregated OOF metrics
    tree_results.append({
        'Model': name,
        # --- Out-of-Fold Regression Metrics ---
        'Train RMSE': np.mean(fold_train_rmse),
        'Test RMSE (OOF)': np.mean(fold_val_rmse),
        'RMSE Gap (Test - Train)': np.mean(fold_val_rmse) - np.mean(fold_train_rmse),
        'Test MAE (OOF)': np.mean(fold_val_mae),
        'Test R² (OOF)': np.mean(fold_val_r2),
        # --- Out-of-Fold Classification Metrics ---
        'Grade Accuracy (OOF)': np.mean(fold_val_acc),
        'Grade Precision (OOF)': np.mean(fold_val_precision),
        'Grade Recall (OOF)': np.mean(fold_val_recall),
        'Grade F1-Score (OOF)': np.mean(fold_val_f1)
    })

# 3. Format and Display Ranked Tree Models DataFrame
tree_results_df = (
    pd.DataFrame(tree_results)
    .sort_values(by='Test RMSE (OOF)')
    .reset_index(drop=True)
)

tree_results_df

,Model,Train RMSE,Test RMSE (OOF),RMSE Gap (Test - Train),Test MAE (OOF),Test R² (OOF),Grade Accuracy (OOF),Grade Precision (OOF),Grade Recall (OOF),Grade F1-Score (OOF)
0,Random Forest,6.292968,7.128956,0.835989,5.662483,0.518242,0.539,0.553840,0.539,0.515762
1,Extra Trees,7.185779,7.508337,0.322559,6.095496,0.466494,0.508,0.547819,0.508,0.463797
2,Decision Tree,6.956815,7.940944,0.984128,6.346169,0.401384,0.471,0.478182,0.471,0.455469


Because tree-based models can capture non-linear interactions, one might expect superior predictive power. However, this added capacity is precisely their vulnerability here: high complexity inherently impairs generalization on small datasets. This structural weakness persists—and often escalates—in extreme ensemble architectures like gradient boosting. Regardless of how aggressively we constrain tree depth or apply regularizers, these models struggle to outperform simpler baselines. Ultimately, the limited dataset imposes an unyielding performance ceiling. To achieve meaningful gains on this problem statement, acquiring a larger, more representative dataset is an indispensable prerequisite.

In [56]:
# 1. Prepare Raw Data: Cast string categoricals to pandas 'category' dtype
X_raw = X.copy()
categorical_cols = binary_columns + ordinal_columns

for col in categorical_cols:
    X_raw[col] = X_raw[col].astype('category')

# 1. Instantiate Regularized Gradient Boosting Models
boosting_models = {
    "CatBoost (Regularized)": CatBoostRegressor(
        iterations=100,
        learning_rate=0.05,
        depth=4,                     # Restrict tree depth
        l2_leaf_reg=5.0,             # L2 regularization coefficient
        subsample=0.8,               # Bagging temperature / row subsampling
        cat_features=tuple(categorical_cols), # Tuple avoids sklearn clone error
        random_state=42,
        verbose=0
    ),
    "LightGBM (Regularized)": LGBMRegressor(
        n_estimators=100,
        learning_rate=0.05,
        max_depth=4,                 # Cap tree depth
        num_leaves=15,               # Restrict max leaf nodes (< 2^max_depth)
        min_child_samples=10,        # Minimum data required in a leaf node
        subsample=0.8,               # Row subsampling ratio
        subsample_freq=1,            # Row subsampling frequency
        colsample_bytree=0.8,        # Feature subsampling ratio per tree
        reg_alpha=0.1,               # L1 regularization (Lasso)
        reg_lambda=1.0,              # L2 regularization (Ridge)
        random_state=42,
        verbose=-1
    ),
    "XGBoost (Regularized)": XGBRegressor(
        n_estimators=100,
        learning_rate=0.05,
        max_depth=4,                 # Limit tree depth
        min_child_weight=3,          # Minimum sum of instance weight in a child
        subsample=0.8,               # Subsample 80% of rows
        colsample_bytree=0.8,        # Subsample 80% of features
        alpha=0.1,                   # L1 regularization term
        reg_lambda=1.0,              # L2 regularization term
        enable_categorical=True,
        tree_method='hist',
        random_state=42,
        verbosity=0
    )
}

boosting_results = []
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# 2. Out-of-Fold Validation Loop using Raw Features (X_raw)
for name, model in boosting_models.items():
    fold_train_rmse, fold_val_rmse, fold_val_mae, fold_val_r2 = [], [], [], []
    fold_val_acc, fold_val_precision, fold_val_recall, fold_val_f1 = [], [], [], []
    
    for train_idx, val_idx in kf.split(X_raw, y):
        X_train, X_val = X_raw.iloc[train_idx], X_raw.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
        
        # Clone model instance for clean fold fitting
        fold_model = clone(model)
        fold_model.fit(X_train, y_train)
        
        # Train RMSE (Overfitting diagnostic)
        y_train_pred = fold_model.predict(X_train)
        fold_train_rmse.append(root_mean_squared_error(y_train, y_train_pred))
        
        # Out-of-Fold Validation Predictions
        y_val_pred = fold_model.predict(X_val)
        
        # Out-of-Fold Regression Metrics
        fold_val_rmse.append(root_mean_squared_error(y_val, y_val_pred))
        fold_val_mae.append(mean_absolute_error(y_val, y_val_pred))
        fold_val_r2.append(r2_score(y_val, y_val_pred))
        
        # Out-of-Fold Converted Grade Classification Metrics
        y_val_true_grades = convert_scores_to_grades(y_val)
        y_val_pred_grades = convert_scores_to_grades(y_val_pred)
        
        acc = accuracy_score(y_val_true_grades, y_val_pred_grades)
        precision, recall, f1, _ = precision_recall_fscore_support(
            y_val_true_grades, 
            y_val_pred_grades, 
            average='weighted', 
            zero_division=0
        )
        
        fold_val_acc.append(acc)
        fold_val_precision.append(precision)
        fold_val_recall.append(recall)
        fold_val_f1.append(f1)
        
    # Fit model on 100% of X_raw for final production storage
    production_model = clone(model)
    production_model.fit(X_raw, y)
    fitted_production_pipelines[name] = production_model
    
    # Store aggregated OOF metrics
    boosting_results.append({
        'Model': name,
        # --- Out-of-Fold Regression Metrics ---
        'Train RMSE': np.mean(fold_train_rmse),
        'Test RMSE (OOF)': np.mean(fold_val_rmse),
        'RMSE Gap (Test - Train)': np.mean(fold_val_rmse) - np.mean(fold_train_rmse),
        'Test MAE (OOF)': np.mean(fold_val_mae),
        'Test R² (OOF)': np.mean(fold_val_r2),
        # --- Out-of-Fold Classification Metrics ---
        'Grade Accuracy (OOF)': np.mean(fold_val_acc),
        'Grade Precision (OOF)': np.mean(fold_val_precision),
        'Grade Recall (OOF)': np.mean(fold_val_recall),
        'Grade F1-Score (OOF)': np.mean(fold_val_f1)
    })

# 3. Format and Display Ranked Master Boosting DataFrame
boosting_results_df = (
    pd.DataFrame(boosting_results)
    .sort_values(by='Test RMSE (OOF)')
    .reset_index(drop=True)
)

boosting_results_df

,Model,Train RMSE,Test RMSE (OOF),RMSE Gap (Test - Train),Test MAE (OOF),Test R² (OOF),Grade Accuracy (OOF),Grade Precision (OOF),Grade Recall (OOF),Grade F1-Score (OOF)
0,LightGBM (Regularized),4.892413,6.460641,1.568227,5.131324,0.604592,0.560,0.565764,0.560,0.551272
1,CatBoost (Regularized),5.810663,6.469915,0.659253,5.148742,0.603487,0.561,0.575520,0.561,0.544988
2,XGBoost (Regularized),4.687551,6.531119,1.843568,5.191462,0.595560,0.555,0.559900,0.555,0.544168


To be honest, the underwhelming performance of the gradient boosting models was completely expected. Boosting algorithms are high-capacity, complex architectures designed for large datasets; on a small sample size, they inevitably overfit. This is clearly demonstrated by our train-test RMSE gaps: boosting models exhibit gaps exceeding $1.0$, whereas traditional linear models maintain tight gaps well below $1.0$. This severe generalization penalty is directly reflected in their lower out-of-fold $R^2$ and accuracy scores, where simpler traditional models actually outperform them. Ultimately, with such a constrained sample size drawn from a much broader population, these empirical results serve as a reminder that model complexity must align with data volume.

In [57]:
def save_model_metadata(metadata_dict, filepath):
    """Converts numpy numeric types to standard Python types and writes JSON."""
    clean_metrics = {}
    for k, v in metadata_dict.get("metrics", {}).items():
        if isinstance(v, (np.floating, np.integer)):
            clean_metrics[k] = v.item()
        elif isinstance(v, (float, int)):
            clean_metrics[k] = v
        else:
            clean_metrics[k] = str(v)
            
    metadata_dict["metrics"] = clean_metrics
    
    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(metadata_dict, f, indent=4)


# 1. Setup target directory
output_dir = "saved_models"
os.makedirs(output_dir, exist_ok=True)


# --- SECTION 1: SAVE BEST TRADITIONAL ML MODEL ---
print("--- 1. Saving Best Traditional ML Model ---")

best_ml_model = traditional_results_df.sort_values(
    by=['Test R² (OOF)', 'Grade Accuracy (OOF)'], 
    ascending=[False, False]
).iloc[0]

best_model_name = str(best_ml_model['Model'])

if best_model_name in fitted_production_pipelines:
    best_pipeline = fitted_production_pipelines[best_model_name]
    trad_model_path = os.path.join(output_dir, "best_traditional_ml_model.joblib")
    trad_meta_path = os.path.join(output_dir, "best_traditional_ml_model_metadata.json")

    # Save Pipeline & Metadata
    joblib.dump(best_pipeline, trad_model_path)
    save_model_metadata({
        "model_name": best_model_name,
        "model_type": "Traditional ML",
        "metrics": best_ml_model.to_dict()
    }, trad_meta_path)

    print(f"✅ Selected & Saved Best Traditional Model: '{best_model_name}'")
    print(f"   └─ Pipeline: {trad_model_path}")
    print(f"   └─ Metadata: {trad_meta_path}")
else:
    print(f"⚠️ Model '{best_model_name}' not found in fitted_production_pipelines.")


# --- SECTION 2: SAVE VOTING REGRESSOR ---
print("\n--- 2. Saving Voting Regressor ---")

voting_model_name = "Voting Regressor"

if voting_model_name in fitted_production_pipelines:
    voting_pipeline = fitted_production_pipelines[voting_model_name]
    voting_model_path = os.path.join(output_dir, "voting_regressor.joblib")
    voting_meta_path = os.path.join(output_dir, "voting_regressor_metadata.json")
    
    # Extract Voting Regressor metrics if available in traditional_results_df
    voting_row = traditional_results_df[
        traditional_results_df['Model'].str.contains('Voting', case=False, na=False)
    ]
    voting_metrics = voting_row.iloc[0].to_dict() if not voting_row.empty else {}
    
    # Save Pipeline & Metadata
    joblib.dump(voting_pipeline, voting_model_path)
    save_model_metadata({
        "model_name": voting_model_name,
        "model_type": "Ensemble (Voting Regressor)",
        "metrics": voting_metrics
    }, voting_meta_path)
    
    print(f"✅ Saved Voting Regressor pipeline to: {voting_model_path}")
    print(f"✅ Saved Voting Regressor metadata to: {voting_meta_path}")
else:
    print("⚠️ 'Voting Regressor' not found in fitted_production_pipelines.")


# --- SECTION 3: SAVE ALL BOOSTING MODELS ---
print("\n--- 3. Saving Boosting Models ---")

for idx, row in boosting_results_df.iterrows():
    model_name = str(row['Model'])
    
    if model_name in fitted_production_pipelines:
        boosting_pipeline = fitted_production_pipelines[model_name]
        
        # Clean string formatting for file path (e.g. "CatBoost (Regularized)" -> "boosting_catboost_regularized")
        clean_name = model_name.lower().replace(" ", "_").replace("(", "").replace(")", "")
        boosting_model_path = os.path.join(output_dir, f"boosting_{clean_name}.joblib")
        boosting_meta_path = os.path.join(output_dir, f"boosting_{clean_name}_metadata.json")
        
        # Save Pipeline & Metadata
        joblib.dump(boosting_pipeline, boosting_model_path)
        save_model_metadata({
            "model_name": model_name,
            "model_type": "Gradient Boosting",
            "metrics": row.to_dict()
        }, boosting_meta_path)
        
        print(f"✅ Saved Boosting Model: '{model_name}' -> {boosting_model_path}")
    else:
        print(f"⚠️ Boosting Model '{model_name}' not found in fitted_production_pipelines.")

--- 1. Saving Best Traditional ML Model ---
✅ Selected & Saved Best Traditional Model: 'Bayesian Ridge'
   └─ Pipeline: saved_models/best_traditional_ml_model.joblib
   └─ Metadata: saved_models/best_traditional_ml_model_metadata.json

--- 2. Saving Voting Regressor ---
✅ Saved Voting Regressor pipeline to: saved_models/voting_regressor.joblib
✅ Saved Voting Regressor metadata to: saved_models/voting_regressor_metadata.json

--- 3. Saving Boosting Models ---
✅ Saved Boosting Model: 'LightGBM (Regularized)' -> saved_models/boosting_lightgbm_regularized.joblib
✅ Saved Boosting Model: 'CatBoost (Regularized)' -> saved_models/boosting_catboost_regularized.joblib
✅ Saved Boosting Model: 'XGBoost (Regularized)' -> saved_models/boosting_xgboost_regularized.joblib


We deliberately omit standalone decision tree models from storage, as their poor performance and significantly lower accuracy make them unsuitable for deployment. Conversely, we persist the boosting models strictly to demonstrate empirical evidence of their failure—serving as a clear benchmark of how complex gradient boosting architectures overfit and struggle when constrained by a small dataset.

# **Model Performance Analysis**